In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
from scipy.optimize import least_squares

from portable_helpers import M_ogse_mixed_offset, M_ogse_free, _roi_color_map, DEFAULT_BRAIN_MARKERS

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

MASTER_PATH  = Path("../../analysis/brains/ogse_experiments/master.long.parquet")
OUT_DIR      = Path("../../analysis/brains/ogse_experiments/lab/fit_rest-tort_CC-testb4")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIRECTIONS  = ["tra", "long"]
N_LIST      = [4, 8]
TD_LIST     = None
# ROI_LIST
#    None → All ROIS
#    ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC"] → CC
#    ["Left-Lateral-Ventricle", "Right-Lateral-Ventricle"] → Ventricles
ROI_LIST    = ["AntCC","MidAntCC","CentralCC","MidPostCC","PostCC","Left-Lateral-Ventricle", "Right-Lateral-Ventricle"]
G_COLUMN    = "g_thorsten"
G_CORRECTION_COLUMN = "grad_correction_factor"
Y_COLUMN    = "value_norm"   # "value" (raw a.u.) or "value_norm" (normalised by S0)
D0_FIXED    = 3.2e-12       # m²/ms, fixed free-water diffusivity

# ALPHA_GLOBAL or TC_GLOBAL: scope for a shared parameter fit
#   None                  → per-curve (one value per td, N, direction)
#   "td"                  → one value per (subj, roi, direction, td), shared across N
#   "td_across_direction" → one value per (subj, roi, td), shared across direction & N
#   "direction"           → one value per (subj, roi, direction), shared across td & N
#   "roi"                 → one value per (subj, roi), shared across td, N & direction
#   "subj"                → one value per subject, shared across all roi, td, N & direction
#   "all"                 → one value for all data

# ── Correlation time tc1 ──────────────────────────────────────────────────────
# TC1_FIXED:
#   None                                    → fitted per curve
#   "master"                                → from master table (TC1_MASTER_COL)
#   float                                   → same tc1 [ms] for all subj/roi/direction/td
#   {subj: float}                           → fixed per subject
#   {subj: {roi: float}}                    → fixed per subject and roi
#   {subj: {roi: {direction: float}}}       → fixed per subject, roi, and direction
#   {subj: {roi: {direction: {td: float}}}} → fixed per subject, roi, direction, and td
#   Any (subj, roi, direction, td) combo absent from the dict is fitted freely —
#   this can mix fixed and free curves within the same GLOBAL joint fit.
TC1_FIXED   = None
TC1_INIT    = 2.0            # ms, initial guess
TC1_BOUNDS  = (0.05, 1000.0) # (lower, upper) ms
TC1_GLOBAL  = None           # None → per-curve; "td_across_direction"/"direction"/"roi"/"subj"/"all" → shared scope
TC1_MASTER_COL = "tc_ms"

# ── Correlation time tc2 ──────────────────────────────────────────────────────
# TC2_FIXED: same shapes as TC1_FIXED (None / "master" / float / nested dict).
TC2_FIXED   = None
TC2_INIT    = 20.0
TC2_BOUNDS  = (0.05, 1000.0)
TC2_GLOBAL  = None
TC2_MASTER_COL = "tc2_ms"

# ── Tortuosity factor α1 ──────────────────────────────────────────────────────
# ALPHA1_FIXED: same shapes as TC1_FIXED (None / "master" / float / nested dict).
ALPHA1_FIXED  = None
ALPHA1_INIT   = 0.5
ALPHA1_BOUNDS = (0.0, 1.0)
ALPHA1_GLOBAL = None
ALPHA1_MASTER_COL = "alpha_macro"

# ── Tortuosity factor α2 ──────────────────────────────────────────────────────
# ALPHA2_FIXED: same shapes as TC1_FIXED (None / "master" / float / nested dict).
ALPHA2_FIXED  = None
ALPHA2_INIT   = 0.5
ALPHA2_BOUNDS = (0.0, 1.0)
ALPHA2_GLOBAL = None
ALPHA2_MASTER_COL = "alpha_macro"

# ── Mixing fractions f1, f2 ───────────────────────────────────────────────────
# Model:  S(G) = M0 · (f1·S1_mixed(tc1,α1,G) + f2·S2_mixed(tc2,α2,G))
# where   M0 = sqrt(S0² − RN²)  (same Rician derivation as single-component model)
#
# F_TOTAL : float → f1 + f2 is constrained to F_TOTAL (f2 = F_TOTAL − f1, not fitted);
#           None  → f1 and f2 are independently free.
#
# F1_FIXED (no "master" option — no master-table column for f1):
#   None                                    → f1 is fitted
#   float                                   → same f1 for all subj/roi/direction/td
#   {subj: float}                           → fixed per subject
#   {subj: {roi: float}}                    → fixed per subject and roi
#   {subj: {roi: {direction: float}}}       → fixed per subject, roi, and direction
#   {subj: {roi: {direction: {td: float}}}} → fixed per subject, roi, direction, and td
#   Any (subj, roi, direction, td) combo absent from the dict is fitted freely —
#   this can mix fixed and free curves within the same GLOBAL joint fit.
#
# F1_GLOBAL: how many curves share the same f1 when it is free. (analogous to TC1_GLOBAL / ALPHA1_GLOBAL)
#   None                  → per-curve (one f1 per td, N, direction)
#   "td"                  → one f1 per (subj, roi, direction, td), shared across N
#   "td_across_direction" → one f1 per (subj, roi, td), shared across direction & N
#   "direction"           → one f1 per (subj, roi, direction), shared across td & N
#   "roi"                 → one f1 per (subj, roi), shared across td, N & direction
#   "subj"                → one f1 per subject
#   "all"                 → one f1 for all data
F_TOTAL   = 1.0    # typically 1.0; set None to let f1 + f2 float freely
F1_FIXED  = None
F1_GLOBAL = "td_across_direction"  # scope for shared f1: one f1 per (subj, roi, td), shared across direction & N

# ── Model name ────────────────────────────────────────────────────────────────
MODEL_NAME = 'mixed-mixed'

# ── Rician noise floor RN (signal units) ─────────────────────────────────────
# Signal model:  S(G) = sqrt( (M0·(f1·S1 + f2·S2))² + RN² )
#                where  M0 = sqrt(S0² − RN²)  [derived from data, not fitted]
#
#   None                        → no noise correction (RN = 0)
#   float                       → same RN for all subjects, td, and roi
#   {td: float}                 → fixed per td, same for all subjects and roi
#   {subj: float}               → fixed per subject, same for all td and roi
#   {subj: {td: float}}         → fixed per subject and td, same for all roi
#   {subj: {td: {roi: float}}}  → fixed per subject, td, and roi

RN_FIXED = {
    "BRAIN": {
        76.0:  {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
        90.0:  {"AntCC": 32, "MidAntCC": 38, "CentralCC": 36, "MidPostCC": 42, "PostCC": 36},
        120.0: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        143.4: {"AntCC": 36, "MidAntCC": 44, "CentralCC": 40, "MidPostCC": 46, "PostCC": 40},
        210.0: {"AntCC": 37, "MidAntCC": 38, "CentralCC": 48, "MidPostCC": 49, "PostCC": 53},
    },
    "LUDG": {
        90.0:  {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
        120.0: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        143.4: {"AntCC": 30, "MidAntCC": 42, "CentralCC": 34, "MidPostCC": 44, "PostCC": 44},
        210.0: {"AntCC": 31, "MidAntCC": 35, "CentralCC": 38, "MidPostCC": 48, "PostCC": 49},
    },
    "MBBL": {
        90.0:  {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
        120.0: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        143.4: {"AntCC": 31, "MidAntCC": 41, "CentralCC": 47, "MidPostCC": 49, "PostCC": 44},
        210.0: {"AntCC": 35, "MidAntCC": 40, "CentralCC": 40, "MidPostCC": 52, "PostCC": 51},
    },
}

# ── Noise-floor truncation ────────────────────────────────────────────────────
# Exclude data points where y < RN × (1 + G_TRUNCATE_EPSILON) from the fit.
# These points are dominated by Rician noise and distort the fitted parameters.
# Plots (Cell 6, Cell 8) are NOT affected — all original data points are shown.
#   None  → no truncation (keep all points, current behaviour)
#   float → e.g. 0.10 excludes points within 10 % above the noise floor
G_TRUNCATE_EPSILON = None

# ── Shared S0 across N values ─────────────────────────────────────────────────
# If |S0(Ni) − S0(Nj)| / S0_avg < S0_SHARE_THRESHOLD for all N pairs at the
# same (subj, roi, direction, td), use S0_avg for all N instead of individual S0.
# Prevents the normalization from amplifying small S0 fluctuations between N.
#   None  → each N uses its own S0 (current behaviour)
#   float → e.g. 0.02 → share S0 if they agree within 2 %
S0_SHARE_THRESHOLD = 0.05

# ── Free-water contamination correction (td=210 reference) ──────────────────
# For each (subj, roi, direction, N), fits a free-diffusion model to that
# ROI's own td=210 curve, then subtracts it — rescaled by the ventricle b=0
# amplitude ratio between each td and 210 — from every OTHER td curve of the
# same ROI/direction/N, before the mixed-mixed fits run. See the dedicated
# cell right after FIT FUNCTIONS.
FREE_WATER_CORRECTION     = True   # flag: True enables the block, False skips it
FREE_WATER_TD_REF         = 210.0
FREE_WATER_VENTRICLE_ROIS = ["Left-Lateral-Ventricle", "Right-Lateral-Ventricle"]
FREE_WATER_D0_INIT        = D0_FIXED
FREE_WATER_D0_BOUNDS      = (1e-13, 1e-11)  # m²/ms
# None       → each direction fits its OWN td=210 curve (per-direction reference)
# "long"/"tra" → force ALL directions to be corrected using the free-water fit
#              from this one direction's td=210 curve (and its ventricle ratio)
FREE_WATER_REF_DIRECTION  = None


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_parquet(MASTER_PATH)

data = df[
    (df.row_kind == "signal_rotated") &
    (df.direction.isin(DIRECTIONS)) &
    (df.N.isin(N_LIST)) &
    (df.stat == "avg")
].copy()

missing_cols = [c for c in [G_COLUMN, Y_COLUMN, G_CORRECTION_COLUMN] if c is not None and c not in data.columns]
if missing_cols:
    raise KeyError(f"Missing column(s): {missing_cols}")

data["G_fit"] = data[G_COLUMN]
if G_CORRECTION_COLUMN is not None:
    data["G_fit"] = data["G_fit"] * data[G_CORRECTION_COLUMN]
data["y_raw"] = data["value"]   # raw signal — always kept to derive S0
data["y_fit"] = data[Y_COLUMN]

G_LABEL = f"Modulation gradient G [mT/m] ({G_COLUMN})"
Y_LABEL = f"{Y_COLUMN} [a.u.]"

if TD_LIST is not None:
    data = data[data.td_ms.isin(TD_LIST)]
if ROI_LIST is not None:
    data = data[data.roi.isin(ROI_LIST)]

print(f"Rows loaded: {len(data)}")
print("Groups (subj, roi, dir):", data.groupby(["subj", "roi", "direction"]).ngroups)
print(f"Gradient column: {G_COLUMN}" + (f" × {G_CORRECTION_COLUMN}" if G_CORRECTION_COLUMN else ""))
print(f"Signal column:   {Y_COLUMN}")
data[["subj", "roi", "direction", "td_ms", "N", "G_fit", "y_fit"]]

Rows loaded: 3982
Groups (subj, roi, dir): 42
Gradient column: g_thorsten × grad_correction_factor
Signal column:   value_norm


,subj,roi,direction,td_ms,N,G_fit,y_fit
5302,BRAIN,AntCC,tra,90.0,4,0.000000,1.000000
5303,BRAIN,AntCC,tra,90.0,4,6.217490,1.006017
5304,BRAIN,AntCC,tra,90.0,4,12.434980,0.921373
5305,BRAIN,AntCC,tra,90.0,4,18.652470,0.960419
5306,BRAIN,AntCC,tra,90.0,4,24.869960,0.884877
...,...,...,...,...,...,...,...
179135,MBBL,Right-Lateral-Ventricle,long,210.0,8,35.087762,0.051134
179136,MBBL,Right-Lateral-Ventricle,long,210.0,8,40.935722,0.037183
179137,MBBL,Right-Lateral-Ventricle,long,210.0,8,46.783683,0.035645
179138,MBBL,Right-Lateral-Ventricle,long,210.0,8,52.631643,0.035446


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT FUNCTIONS
#
# Signal model:  S(G) = sqrt( (M0·(f1·S1 + f2·S2))² + RN² )
#                where  M0 = sqrt(S0² − RN²)  [derived from data, not fitted]
#                S1 = S_mixed(tc1, α1, G),  S2 = S_mixed(tc2, α2, G)
#                f1 + f2 = F_TOTAL  if F_TOTAL is not None, else f1, f2 are independent
# ══════════════════════════════════════════════════════════════════════════════

def _resolve_rn(subj, td, roi=None):
    """Return fixed RN for (subj, td, roi). Returns 0.0 when RN_FIXED is None."""
    if RN_FIXED is None:
        return 0.0
    if isinstance(RN_FIXED, dict):
        first_key = next(iter(RN_FIXED))
        if isinstance(first_key, str):               # {subj: ...}
            subj_entry = RN_FIXED.get(subj)
            if subj_entry is None:
                return 0.0
            if isinstance(subj_entry, dict):         # {subj: {td: ...}}
                td_entry = subj_entry.get(float(td), subj_entry.get(td))
                if td_entry is None:
                    return 0.0
                if isinstance(td_entry, dict):       # {subj: {td: {roi: float}}}
                    return float(td_entry.get(roi, 0.0))
                return float(td_entry)               # {subj: {td: float}}
            return float(subj_entry)                 # {subj: float}
        return float(RN_FIXED.get(float(td), RN_FIXED.get(td, 0.0)))  # {td: float}
    return float(RN_FIXED)


def _m0_from_s0(s0_raw, rn):
    """M0 = sqrt(S0² − RN²). For value_norm returns M0/S0."""
    m0 = np.sqrt(max(float(s0_raw)**2 - float(rn)**2, 0.0))
    if Y_COLUMN == "value_norm":
        return m0 / float(s0_raw) if float(s0_raw) > 0 else 1.0
    return m0


def _get_master_value(subj, roi, direction, col):
    """Look up a fixed value from the master table for one (subj, roi, direction) group."""
    rows = df[(df.subj == subj) & (df.roi == roi) & (df.direction == direction)]
    if col in rows.columns and not rows[col].isna().all():
        return float(rows[col].dropna().iloc[0])
    raise KeyError(f"'{col}' not found or all-NaN for {subj}/{roi}/{direction}")


def _resolve_dict_fixed(fixed, subj, roi, direction, td):
    """Walk a nested dict keyed subj → roi → direction → td (each level optional —
    hitting a non-dict value fixes it for everything below). Returns the fixed
    float, or None if this combo isn't covered (→ fitted freely)."""
    val = fixed
    for key in (subj, roi, direction):
        if not isinstance(val, dict):
            return float(val)
        if key not in val:
            return None
        val = val[key]
    if isinstance(val, dict):
        td_f = float(td)
        if td_f in val:
            val = val[td_f]
        elif td in val:
            val = val[td]
        else:
            return None
    return None if isinstance(val, dict) else float(val)


def _resolve_tc1(subj, roi, direction, td):
    """Return fixed tc1 [ms], or None (→ fitted)."""
    if TC1_FIXED is None:
        return None
    if TC1_FIXED == "master":
        return _get_master_value(subj, roi, direction, TC1_MASTER_COL)
    if isinstance(TC1_FIXED, dict):
        return _resolve_dict_fixed(TC1_FIXED, subj, roi, direction, td)
    return float(TC1_FIXED)


def _resolve_tc2(subj, roi, direction, td):
    """Return fixed tc2 [ms], or None (→ fitted)."""
    if TC2_FIXED is None:
        return None
    if TC2_FIXED == "master":
        return _get_master_value(subj, roi, direction, TC2_MASTER_COL)
    if isinstance(TC2_FIXED, dict):
        return _resolve_dict_fixed(TC2_FIXED, subj, roi, direction, td)
    return float(TC2_FIXED)


def _resolve_alpha1(subj, roi, direction, td):
    """Return fixed α1, or None (→ fitted)."""
    if ALPHA1_FIXED is None:
        return None
    if ALPHA1_FIXED == "master":
        return _get_master_value(subj, roi, direction, ALPHA1_MASTER_COL)
    if isinstance(ALPHA1_FIXED, dict):
        return _resolve_dict_fixed(ALPHA1_FIXED, subj, roi, direction, td)
    return float(ALPHA1_FIXED)


def _resolve_alpha2(subj, roi, direction, td):
    """Return fixed α2, or None (→ fitted)."""
    if ALPHA2_FIXED is None:
        return None
    if ALPHA2_FIXED == "master":
        return _get_master_value(subj, roi, direction, ALPHA2_MASTER_COL)
    if isinstance(ALPHA2_FIXED, dict):
        return _resolve_dict_fixed(ALPHA2_FIXED, subj, roi, direction, td)
    return float(ALPHA2_FIXED)


def _resolve_f1(subj, roi, direction, td):
    """Return fixed f1, or None (→ fitted). No 'master' option — no master column for f1."""
    if F1_FIXED is None:
        return None
    if isinstance(F1_FIXED, dict):
        return _resolve_dict_fixed(F1_FIXED, subj, roi, direction, td)
    return float(F1_FIXED)


def _predict(td, G, N, tc1, alpha1, tc2, alpha2, f1, f2, M0, rn):
    """S(G) = sqrt( (M0·(f1·S1 + f2·S2))² + RN² )"""
    with np.errstate(over="ignore", invalid="ignore"):
        s1  = M_ogse_mixed_offset(td, G, N, td / N, tc1, alpha1, 1, D0_FIXED, 0, 0)
        s2  = M_ogse_mixed_offset(td, G, N, td / N, tc2, alpha2, 1, D0_FIXED, 0, 0)
        sig = M0 * (f1 * s1 + f2 * s2)
        return np.sqrt(sig**2 + rn**2) if rn != 0.0 else sig


def _param_stderr(result, param_idx):
    """1-σ error on params[param_idx] from the least-squares Jacobian."""
    dof = result.fun.size - result.x.size
    if dof <= 0 or result.x.size == 0 or param_idx >= result.x.size:
        return np.nan
    try:
        cov = np.linalg.pinv(result.jac.T @ result.jac) * (2.0 * result.cost / dof)
        return float(np.sqrt(max(float(cov[param_idx, param_idx]), 0.0)))
    except np.linalg.LinAlgError:
        return np.nan


def fit_curve(td, N, G, y, subj=None, roi=None, direction=None, s0=None):
    """Fit tc1, tc2, α1, α2, f1 for ONE (td, N, direction) curve.

    Returns (tc1, tc2, alpha1, alpha2, f1, f2, result, free).
    tc1, tc2 are optimised in log-space for numerical stability.
    f1 is bounded in [0, F_TOTAL] when F_TOTAL is not None, else [0, 1].
    When F_TOTAL is not None: f2 = F_TOTAL − f1 (not a free parameter).
    When F_TOTAL is None: f2 is fitted independently in [0, 1].
    `free` is the ordered list of parameter names actually optimised (their
    position in `free` matches their index in `result.x`), needed by callers
    to extract per-parameter stderrs since which params are fixed can now
    vary per curve (dict-based *_FIXED).
    """
    rn = _resolve_rn(subj, td, roi)
    M0 = _m0_from_s0(s0, rn) if s0 is not None else float(y.max())

    tc1_fv  = _resolve_tc1(subj, roi, direction, td)
    tc2_fv  = _resolve_tc2(subj, roi, direction, td)
    al1_fv  = _resolve_alpha1(subj, roi, direction, td)
    al2_fv  = _resolve_alpha2(subj, roi, direction, td)
    f1_fv   = _resolve_f1(subj, roi, direction, td)
    f_total = float(F_TOTAL)  if F_TOTAL  is not None else None

    free, x0, lo, hi = [], [], [], []
    if tc1_fv is None:
        free.append("tc1")
        x0.append(np.log(TC1_INIT)); lo.append(np.log(TC1_BOUNDS[0])); hi.append(np.log(TC1_BOUNDS[1]))
    if tc2_fv is None:
        free.append("tc2")
        x0.append(np.log(TC2_INIT)); lo.append(np.log(TC2_BOUNDS[0])); hi.append(np.log(TC2_BOUNDS[1]))
    if al1_fv is None:
        free.append("alpha1")
        x0.append(ALPHA1_INIT); lo.append(ALPHA1_BOUNDS[0]); hi.append(ALPHA1_BOUNDS[1])
    if al2_fv is None:
        free.append("alpha2")
        x0.append(ALPHA2_INIT); lo.append(ALPHA2_BOUNDS[0]); hi.append(ALPHA2_BOUNDS[1])
    if f1_fv is None:
        free.append("f1")
        f1_hi = f_total if f_total is not None else 1.0
        x0.append(0.5 * f1_hi); lo.append(0.0); hi.append(f1_hi)
    if f_total is None:  # f2 is also free (independent of f1)
        free.append("f2")
        f2_init = 1.0 - (f1_fv if f1_fv is not None else 0.5)
        x0.append(max(0.0, f2_init)); lo.append(0.0); hi.append(1.0)

    def _unpack(params):
        idx = 0
        if "tc1" in free:
            tc1 = np.exp(params[idx]); idx += 1
        else:
            tc1 = tc1_fv
        if "tc2" in free:
            tc2 = np.exp(params[idx]); idx += 1
        else:
            tc2 = tc2_fv
        if "alpha1" in free:
            alpha1 = params[idx]; idx += 1
        else:
            alpha1 = al1_fv
        if "alpha2" in free:
            alpha2 = params[idx]; idx += 1
        else:
            alpha2 = al2_fv
        if "f1" in free:
            f1 = params[idx]; idx += 1
        else:
            f1 = f1_fv
        if f_total is not None:
            f2 = f_total - f1
        elif "f2" in free:
            f2 = params[idx]
        else:
            f2 = 1.0 - f1
        return tc1, tc2, alpha1, alpha2, f1, f2

    if not free:
        tc1, tc2, alpha1, alpha2, f1, f2 = _unpack([])
        fun = _predict(td, G, N, tc1, alpha1, tc2, alpha2, f1, f2, M0, rn) - y
        mock = type("FixedResult", (), {
            "x": np.array([]), "fun": fun, "jac": np.zeros((len(fun), 0)),
            "cost": 0.5 * float(np.sum(fun**2)), "success": True,
        })()
        return tc1, tc2, alpha1, alpha2, f1, f2, mock, free

    def residuals(params):
        tc1, tc2, alpha1, alpha2, f1, f2 = _unpack(params)
        out = _predict(td, G, N, tc1, alpha1, tc2, alpha2, f1, f2, M0, rn) - y
        return np.where(np.isfinite(out), out, 1e6)

    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)
    tc1, tc2, alpha1, alpha2, f1, f2 = _unpack(result.x)
    return tc1, tc2, alpha1, alpha2, f1, f2, result, free


# ── Scope definitions for global fitting ─────────────────────────────────────
_SCOPE_COL_ORDER = ["subj", "roi", "direction", "td_ms"]
_SCOPE_COLS = {
    "all":                 [],
    "subj":                ["subj"],
    "roi":                 ["subj", "roi"],
    "direction":           ["subj", "roi", "direction"],
    "td":                  ["subj", "roi", "direction", "td_ms"],
    "td_across_direction": ["subj", "roi", "td_ms"],
}


def _group_scope():
    """Return (scope_name, group_cols) for the outer split that gathers every
    curve any active shared parameter spans.

    Each active _GLOBAL scope needs every curve sharing its key to land in the
    same joint least_squares call. Grouping the outer split on columns common
    to ALL active scopes (the intersection of their _SCOPE_COLS) guarantees
    that: a curve's outer-group key is then a function of every active scope's
    key, so curves that share a scope key always share the outer-group key too.
    Scopes need not form a strict hierarchy — e.g. "td_across_direction" and
    "direction" both refine "roi" but neither refines the other; their
    intersection ["subj", "roi"] is exactly the "roi" scope. Parameters whose
    own scope is finer than this outer split are handled by giving them one
    shared index per distinct sub-key inside fit_group_global — NOT by
    shrinking the outer split down to the finest scope.
    """
    candidates = [s for s in (TC1_GLOBAL, TC2_GLOBAL, ALPHA1_GLOBAL, ALPHA2_GLOBAL, F1_GLOBAL) if s is not None]
    if not candidates:
        return "all", []
    common = set(_SCOPE_COLS[candidates[0]])
    for s in candidates[1:]:
        common &= set(_SCOPE_COLS[s])
    scope_cols = [c for c in _SCOPE_COL_ORDER if c in common]
    scope_name = "+".join(scope_cols) if scope_cols else "all"
    return scope_name, scope_cols


def fit_group_global(group_df):
    """Joint fit across all curves in group_df.

    Each of tc1, tc2, alpha1, alpha2, f1 is shared according to its OWN _GLOBAL
    scope: one fitted value per distinct key of _SCOPE_COLS[GLOBAL] present in
    group_df — not necessarily one value for the whole group_df. This lets
    different parameters use different sharing granularities within the same
    joint least_squares call (e.g. tc1 shared over a whole direction while f1
    gets one value per td within that direction).

    Each parameter is resolved PER CURVE via its _resolve_* function: a curve
    is fixed (not optimised) when that combo is covered by its *_FIXED value
    (float / "master" / dict entry), and free otherwise. Since a dict may
    cover only some combos, fixed and free curves can be mixed within the
    same shared-scope group and the same joint least_squares call.

    Returns: (result, curves,
              per_tc1s, per_tc1_errs, per_tc2s, per_tc2_errs,
              per_al1s, per_al1_errs, per_al2s, per_al2_errs,
              per_f1s, per_f1_errs, per_f2s, M0s)
    tc errors are in ms (converted from log-space Jacobian: σ_tc = tc · σ_{log_tc}).
    """
    f_total = float(F_TOTAL) if F_TOTAL is not None else None

    curves = []
    for (subj, roi, direction), dir_grp in group_df.groupby(["subj", "roi", "direction"]):
        for td in sorted(float(t) for t in dir_grp.td_ms.unique()):
            sub = dir_grp[np.isclose(dir_grp.td_ms.astype(float), td)]
            for N in N_LIST:
                s = sub[sub.N == N].sort_values("G_fit")
                if s.empty:
                    continue
                s0_raw = float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
                rn = _resolve_rn(subj, td, roi)
                curves.append(dict(
                    subj=subj, roi=roi, direction=direction,
                    td=td, N=int(N), S0=s0_raw,
                    G=s.G_fit.values, y=s.y_fit.values,
                    M0=_m0_from_s0(s0_raw, rn),
                    rn=rn,
                    tc1_fv=_resolve_tc1(subj, roi, direction, td),
                    tc2_fv=_resolve_tc2(subj, roi, direction, td),
                    al1_fv=_resolve_alpha1(subj, roi, direction, td),
                    al2_fv=_resolve_alpha2(subj, roi, direction, td),
                    f1_fv=_resolve_f1(subj, roi, direction, td),
                ))

    M0s = {(c["subj"], c["roi"], c["direction"], c["td"], c["N"]): c["M0"] for c in curves}

    def _curve_key(scope, c):
        lut = {"subj": c["subj"], "roi": c["roi"], "direction": c["direction"], "td_ms": c["td"]}
        return tuple(lut[col] for col in _SCOPE_COLS[scope])

    x0, lo, hi = [], [], []

    def _add_param(fixed_per_curve, global_scope, init, lo_b, hi_b):
        """Per curve: if fixed_per_curve[i] is not None, that curve is pinned
        (no x0 entry). Otherwise it shares an x0 entry with other free curves
        that have the same global_scope key (or gets its own when global_scope
        is None). Returns a per-curve list of ("fixed", value) / ("free", idx)."""
        key_to_idx = {}
        entries = []
        for i, c in enumerate(curves):
            fv = fixed_per_curve[i]
            if fv is not None:
                entries.append(("fixed", fv))
                continue
            key = _curve_key(global_scope, c) if global_scope is not None else i
            if key not in key_to_idx:
                key_to_idx[key] = len(x0)
                x0.append(init); lo.append(lo_b); hi.append(hi_b)
            entries.append(("free", key_to_idx[key]))
        return entries

    tc1_entries = _add_param([c["tc1_fv"] for c in curves], TC1_GLOBAL,
                              np.log(TC1_INIT), np.log(TC1_BOUNDS[0]), np.log(TC1_BOUNDS[1]))
    tc2_entries = _add_param([c["tc2_fv"] for c in curves], TC2_GLOBAL,
                              np.log(TC2_INIT), np.log(TC2_BOUNDS[0]), np.log(TC2_BOUNDS[1]))
    al1_entries = _add_param([c["al1_fv"] for c in curves], ALPHA1_GLOBAL,
                              ALPHA1_INIT, ALPHA1_BOUNDS[0], ALPHA1_BOUNDS[1])
    al2_entries = _add_param([c["al2_fv"] for c in curves], ALPHA2_GLOBAL,
                              ALPHA2_INIT, ALPHA2_BOUNDS[0], ALPHA2_BOUNDS[1])

    f1_hi = f_total if f_total is not None else 1.0
    f1_entries = _add_param([c["f1_fv"] for c in curves], F1_GLOBAL, 0.5 * f1_hi, 0.0, f1_hi)

    f2_entries = None
    if f_total is None:  # f2 always free, per curve, when F_TOTAL is None
        f2_entries = _add_param([None] * len(curves), None, 0.5, 0.0, 1.0)

    def _curve_params(params, i, c):
        kind, val = tc1_entries[i]
        tc1 = np.exp(params[val]) if kind == "free" else val

        kind, val = tc2_entries[i]
        tc2 = np.exp(params[val]) if kind == "free" else val

        kind, val = al1_entries[i]
        alpha1 = params[val] if kind == "free" else val

        kind, val = al2_entries[i]
        alpha2 = params[val] if kind == "free" else val

        kind, val = f1_entries[i]
        f1 = params[val] if kind == "free" else val

        if f_total is not None:
            f2 = f_total - f1
        elif f2_entries is not None:
            f2 = params[f2_entries[i][1]]
        else:
            f2 = 1.0 - f1

        return tc1, tc2, alpha1, alpha2, f1, f2

    if not x0:
        # Every curve is fixed for every parameter (only reachable when
        # f_total is not None — otherwise f2 always adds x0 entries).
        f1_vals, f2_vals, fun_parts = [], [], []
        for c in curves:
            f1_val = c["f1_fv"]
            f2_val = f_total - f1_val
            f1_vals.append(f1_val); f2_vals.append(f2_val)
            fun_parts.append(_predict(c["td"], c["G"], c["N"],
                                       c["tc1_fv"], c["al1_fv"], c["tc2_fv"], c["al2_fv"],
                                       f1_val, f2_val, c["M0"], c["rn"]) - c["y"])
        fun = np.concatenate(fun_parts)
        result = type("FixedResult", (), {
            "x": np.array([]), "fun": fun, "jac": np.zeros((len(fun), 0)),
            "cost": 0.5 * float(np.sum(fun**2)), "success": True,
        })()
        return (result, curves,
                [c["tc1_fv"] for c in curves], [np.nan] * len(curves),
                [c["tc2_fv"] for c in curves], [np.nan] * len(curves),
                [c["al1_fv"] for c in curves], [np.nan] * len(curves),
                [c["al2_fv"] for c in curves], [np.nan] * len(curves),
                f1_vals, [np.nan] * len(curves),
                f2_vals, M0s)

    def residuals(params):
        res = []
        for i, c in enumerate(curves):
            tc1, tc2, alpha1, alpha2, f1, f2 = _curve_params(params, i, c)
            out = _predict(c["td"], c["G"], c["N"], tc1, alpha1, tc2, alpha2, f1, f2, c["M0"], c["rn"]) - c["y"]
            res.append(np.where(np.isfinite(out), out, 1e6))
        return np.concatenate(res)

    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)

    per_tc1s, per_tc1e = [], []
    per_tc2s, per_tc2e = [], []
    per_al1s, per_al1e = [], []
    per_al2s, per_al2e = [], []
    per_f1s,  per_f1e  = [], []
    per_f2s            = []

    for i, c in enumerate(curves):
        tc1, tc2, alpha1, alpha2, f1, f2 = _curve_params(result.x, i, c)
        per_tc1s.append(tc1);   per_tc2s.append(tc2)
        per_al1s.append(alpha1); per_al2s.append(alpha2)
        per_f1s.append(f1);      per_f2s.append(f2)

        kind, val = tc1_entries[i]
        per_tc1e.append(tc1 * _param_stderr(result, val) if kind == "free" else np.nan)
        kind, val = tc2_entries[i]
        per_tc2e.append(tc2 * _param_stderr(result, val) if kind == "free" else np.nan)
        kind, val = al1_entries[i]
        per_al1e.append(_param_stderr(result, val) if kind == "free" else np.nan)
        kind, val = al2_entries[i]
        per_al2e.append(_param_stderr(result, val) if kind == "free" else np.nan)
        kind, val = f1_entries[i]
        per_f1e.append(_param_stderr(result, val) if kind == "free" else np.nan)

    return (result, curves,
            per_tc1s, per_tc1e, per_tc2s, per_tc2e,
            per_al1s, per_al1e, per_al2s, per_al2e,
            per_f1s, per_f1e, per_f2s, M0s)

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# FREE-WATER CONTAMINATION CORRECTION  (td=210 reference)
#
# For each (subj, roi, N) — excluding the ventricle ROIs themselves — fit a
# free-diffusion model (M_ogse_free, with the Rician noise floor) to the
# td=210 curve of the REFERENCE direction (FREE_WATER_REF_DIRECTION, or the
# curve's own direction when None). This isolates the free-water (CSF
# partial-volume) component cleanly, since by td=210 it has fully decayed
# while the tissue-restricted component has barely moved. Then, for every
# direction actually being corrected and every OTHER td (skipping only the
# reference curve itself, i.e. td=210 of the reference direction), evaluate
# that fitted curve at the target curve's G values and subtract it, rescaled
# by the ventricle's b=0 (G≈0) amplitude ratio M_ventr(td, b=0) / M_ventr(210,
# b=0) — computed from the same reference direction — a proxy for the
# T2-driven amplitude change between echo times. Everything downstream (fit
# loop, contrast fits, plots) then runs on this corrected signal unchanged.
#
# QC: the fitted (M0, D0) of every free-water fit, plus a data-vs-fit plot,
# are saved to OUT_DIR/free_water_fits/ so the fits can be checked (D0 should
# sit close to D0_FIXED) before trusting the subtraction.
# ══════════════════════════════════════════════════════════════════════════════

def _s0_raw(df_curve):
    """b=0 (min |G_fit|) raw amplitude of one curve's rows."""
    row = df_curve.loc[df_curve.G_fit.abs().idxmin()]
    return float(row.y_raw)


def _fit_free_water(td, N, G, y, rn):
    """Fit sqrt(M_ogse_free(td,G,N,x,M0,D0)**2 + rn**2) ~= y. Returns (M0, D0, result)."""
    x = td / N

    def residuals(params):
        M0, D0 = params
        model = M_ogse_free(td, G, N, x, M0, D0)
        pred = np.sqrt(model**2 + rn**2) if rn != 0.0 else model
        return pred - y

    x0 = [float(y.max()), FREE_WATER_D0_INIT]
    lo = [0.0, FREE_WATER_D0_BOUNDS[0]]
    hi = [np.inf, FREE_WATER_D0_BOUNDS[1]]
    result = least_squares(residuals, x0, bounds=(lo, hi), method="trf", max_nfev=10000)
    return float(result.x[0]), float(result.x[1]), result


def _s0_by_td(ventr_grp, tds):
    """Ventricle b=0 amplitude per td, averaged over the ventricle ROI(s) present."""
    s0 = {}
    for td in tds:
        vals = [
            _s0_raw(ventr_grp[(ventr_grp.roi == r) & np.isclose(ventr_grp.td_ms.astype(float), td)])
            for r in FREE_WATER_VENTRICLE_ROIS
            if not ventr_grp[(ventr_grp.roi == r) & np.isclose(ventr_grp.td_ms.astype(float), td)].empty
        ]
        if vals:
            s0[td] = float(np.mean(vals))
    return s0


free_water_fits = []  # QC rows: one per (subj, roi, ref_direction, N) free-water fit

if FREE_WATER_CORRECTION:
    FREE_WATER_DIR = OUT_DIR / "free_water_fits"
    FREE_WATER_DIR.mkdir(parents=True, exist_ok=True)

    data_corrected = data.copy()

    for (subj, N), grp in data.groupby(["subj", "N"]):
        directions_present = sorted(grp.direction.unique())
        ref_directions_needed = {
            (FREE_WATER_REF_DIRECTION if FREE_WATER_REF_DIRECTION is not None else d)
            for d in directions_present
        }

        fits_cache = {}      # (ref_direction, roi) -> (M0_fw, D0_fw)
        s0_ventr_cache = {}  # ref_direction -> {td: s0_ventr}

        # ── one free-water fit per (roi, ref_direction) actually needed ─────────
        for ref_direction in ref_directions_needed:
            ref_src_grp = grp[grp.direction == ref_direction]
            ventr = ref_src_grp[ref_src_grp.roi.isin(FREE_WATER_VENTRICLE_ROIS)]
            tds_ref = sorted(float(t) for t in ref_src_grp.td_ms.unique())
            s0_ventr = _s0_by_td(ventr, tds_ref)
            s0_ventr_cache[ref_direction] = s0_ventr

            if FREE_WATER_TD_REF not in s0_ventr:
                print(f"[free-water] {subj}/N={N}: no ventricle b=0 at td={FREE_WATER_TD_REF} "
                      f"for reference direction={ref_direction!r}, skipping")
                continue

            region_rois = [r for r in ref_src_grp.roi.unique() if r not in FREE_WATER_VENTRICLE_ROIS]
            for roi in region_rois:
                ref = ref_src_grp[
                    (ref_src_grp.roi == roi) & np.isclose(ref_src_grp.td_ms.astype(float), FREE_WATER_TD_REF)
                ].sort_values("G_fit")
                if ref.empty:
                    continue

                rn_ref = _resolve_rn(subj, FREE_WATER_TD_REF, roi)
                G_ref, y_ref = ref.G_fit.values, ref.y_raw.values
                M0_fw, D0_fw, fw_result = _fit_free_water(FREE_WATER_TD_REF, N, G_ref, y_ref, rn_ref)
                fits_cache[(ref_direction, roi)] = (M0_fw, D0_fw)

                free_water_fits.append(dict(
                    subj=subj, roi=roi, ref_direction=ref_direction, N=N,
                    td_ref=FREE_WATER_TD_REF, M0_fw=M0_fw, D0_fw=D0_fw,
                    D0_fixed=D0_FIXED, D0_ratio=D0_fw / D0_FIXED,
                    rn=rn_ref, cost=float(fw_result.cost), success=bool(fw_result.success),
                ))

                # ── QC plot: data vs fitted free-water curve at td=210 ───────────
                fig, ax = plt.subplots(figsize=(5, 4))
                ax.plot(G_ref, y_ref, "o", label=f"data (td=210, dir={ref_direction})")
                G_dense = np.linspace(G_ref.min(), G_ref.max(), 200)
                model_dense = M_ogse_free(FREE_WATER_TD_REF, G_dense, N, FREE_WATER_TD_REF / N, M0_fw, D0_fw)
                pred_dense = np.sqrt(model_dense**2 + rn_ref**2) if rn_ref != 0.0 else model_dense
                ax.plot(G_dense, pred_dense, "-", label=f"free-water fit (D0={D0_fw:.2e})")
                ax.set_xlabel(G_LABEL)
                ax.set_ylabel(Y_LABEL)
                ax.set_title(f"{subj} · {roi} · ref={ref_direction} · N={N}")
                ax.legend()
                fig.tight_layout()
                fig.savefig(FREE_WATER_DIR / f"{subj}_{roi}_{ref_direction}_N{N}.png", dpi=120)
                plt.close(fig)

        # ── apply the (possibly shared) fit to every direction being corrected ──
        for direction in directions_present:
            ref_direction = FREE_WATER_REF_DIRECTION if FREE_WATER_REF_DIRECTION is not None else direction
            s0_ventr = s0_ventr_cache.get(ref_direction, {})
            if FREE_WATER_TD_REF not in s0_ventr:
                continue  # already warned above when building s0_ventr_cache

            dir_grp = grp[grp.direction == direction]
            tds = sorted(float(t) for t in dir_grp.td_ms.unique())
            region_rois = [r for r in dir_grp.roi.unique() if r not in FREE_WATER_VENTRICLE_ROIS]

            for roi in region_rois:
                key = (ref_direction, roi)
                if key not in fits_cache:
                    continue
                M0_fw, D0_fw = fits_cache[key]
                roi_grp = dir_grp[dir_grp.roi == roi]

                for td in tds:
                    # Skip only the reference curve itself (its own direction, at td=210) —
                    # every other (direction, td) combo — including this direction's own
                    # td=210 when direction != ref_direction — gets corrected.
                    if direction == ref_direction and np.isclose(td, FREE_WATER_TD_REF):
                        continue
                    if td not in s0_ventr:
                        continue
                    ratio = s0_ventr[td] / s0_ventr[FREE_WATER_TD_REF]

                    curve = roi_grp[np.isclose(roi_grp.td_ms.astype(float), td)]
                    if curve.empty:
                        continue
                    idx = curve.index
                    G_td = data_corrected.loc[idx, "G_fit"].values
                    fw_pred = M_ogse_free(td, G_td, N, td / N, M0_fw, D0_fw)
                    corrected = np.clip(data_corrected.loc[idx, "y_raw"].values - ratio * fw_pred, 0.0, None)

                    data_corrected.loc[idx, "y_raw"] = corrected
                    data_corrected.loc[idx, "value"] = corrected
                    if Y_COLUMN == "value":
                        data_corrected.loc[idx, "y_fit"] = corrected
                    elif Y_COLUMN == "value_norm":
                        s0_new = _s0_raw(data_corrected.loc[idx])
                        data_corrected.loc[idx, "y_fit"] = corrected / s0_new
                    else:
                        raise NotImplementedError(
                            f"FREE_WATER_CORRECTION only supports Y_COLUMN 'value' or 'value_norm', got {Y_COLUMN!r}"
                        )

    free_water_fits_df = pd.DataFrame(free_water_fits)
    free_water_fits_df.to_csv(FREE_WATER_DIR / "free_water_fits.csv", index=False)

    data = data_corrected
    print(
        f"Free-water contamination correction (td=210 reference, "
        f"ref_direction={FREE_WATER_REF_DIRECTION!r}) applied. "
        f"{len(free_water_fits_df)} free-water fits saved to {FREE_WATER_DIR}"
    )
else:
    print("Free-water contamination correction disabled.")


/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  

Free-water contamination correction (td=210 reference, ref_direction=None) applied. 60 free-water fits saved to ../../analysis/brains/ogse_experiments/lab/fit_rest-tort_CC-testb4/free_water_fits


/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: invalid value encountered in divide
  data_corrected.loc[idx, "y_fit"] = corrected / s0_new
/tmp/ipykernel_3215111/3256203658.py:168: RuntimeWarning: divide by zero encountered in divide
  d

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT LOOP  —  runs all (subj, roi, dir, td, N) groups
# ══════════════════════════════════════════════════════════════════════════════

rows      = []
fit_store = {}


def _tc_err(result, tc, pidx):
    return tc * _param_stderr(result, pidx) if pidx is not None else np.nan

def _scalar_err(result, pidx):
    return _param_stderr(result, pidx) if pidx is not None else np.nan


_any_global = any(v is not None for v in [TC1_GLOBAL, TC2_GLOBAL, ALPHA1_GLOBAL, ALPHA2_GLOBAL, F1_GLOBAL])

if not _any_global:
    # ── Per-curve fit: one (tc1, tc2, α1, α2, f1) per (subj, roi, dir, td, N) ──
    for (subj, roi, direction), grp in data.groupby(["subj", "roi", "direction"]):
        tds = sorted(float(t) for t in grp.td_ms.unique())
        fit_store[(subj, roi, direction)] = {"tds": tds, "fits": {td: {} for td in tds}}

        for td in tds:
            sub = grp[np.isclose(grp.td_ms.astype(float), td)]

            # ── Pre-pass: collect S0 per N, optionally share ─────────────────
            s0_per_N = {}
            for _N in N_LIST:
                _s = sub[sub.N == _N].sort_values("G_fit")
                if not _s.empty:
                    s0_per_N[_N] = float(
                        _s.loc[_s.G_fit.abs() == _s.G_fit.abs().min(), "y_raw"].iloc[0]
                    )
            if S0_SHARE_THRESHOLD is not None and len(s0_per_N) >= 2:
                _s0_vals = list(s0_per_N.values())
                _s0_avg  = float(np.mean(_s0_vals))
                _max_dev = max(abs(v - _s0_avg) for v in _s0_vals)
                if _s0_avg > 0 and _max_dev / _s0_avg < S0_SHARE_THRESHOLD:
                    s0_per_N = {_N: _s0_avg for _N in s0_per_N}

            for N in N_LIST:
                s = sub[sub.N == N].sort_values("G_fit")
                if s.empty:
                    continue
                G, y = s.G_fit.values, s.y_fit.values
                s0_raw = s0_per_N.get(
                    N,
                    float(s.loc[s.G_fit.abs() == s.G_fit.abs().min(), "y_raw"].iloc[0])
                )

                # ── Truncate noise-floor points (fit only, plots unchanged) ──
                G_fit, y_fit = G, y
                if G_TRUNCATE_EPSILON is not None:
                    _rn_trunc = _resolve_rn(subj, td, roi)
                    if _rn_trunc > 0:
                        _keep = y > _rn_trunc * (1.0 + G_TRUNCATE_EPSILON)
                        G_fit = G[_keep]
                        y_fit = y[_keep]
                if len(G_fit) == 0:
                    continue

                tc1, tc2, alpha1, alpha2, f1, f2, result, free = fit_curve(
                    td, N, G_fit, y_fit, subj=subj, roi=roi, direction=direction, s0=s0_raw
                )
                tc1_pi = free.index("tc1")    if "tc1"    in free else None
                tc2_pi = free.index("tc2")    if "tc2"    in free else None
                al1_pi = free.index("alpha1") if "alpha1" in free else None
                al2_pi = free.index("alpha2") if "alpha2" in free else None
                f1_pi  = free.index("f1")     if "f1"     in free else None
                rn = _resolve_rn(subj, td, roi)
                M0 = _m0_from_s0(s0_raw, rn)

                print(
                    f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  N={N}  "
                    f"tc1={tc1:.3f} ms  tc2={tc2:.3f} ms  "
                    f"α1={alpha1:.4f}  α2={alpha2:.4f}  "
                    f"f1={f1:.3f}  M0={M0:.4f}  RN={rn:.2f}  cost={result.cost:.3e}"
                )
                fit_store[(subj, roi, direction)]["fits"][td][N] = dict(
                    G=G, y=y, tc1=tc1, tc2=tc2, alpha1=alpha1, alpha2=alpha2,
                    f1=f1, f2=f2, M0=M0, RN=rn, S0=s0_raw,
                )
                rows.append(dict(
                    subj=subj, roi=roi, direction=direction, td_ms=td, N=N,
                    tc1_ms=tc1, tc1_err=_tc_err(result, tc1, tc1_pi),
                    tc2_ms=tc2, tc2_err=_tc_err(result, tc2, tc2_pi),
                    alpha1=alpha1, alpha1_err=_scalar_err(result, al1_pi),
                    alpha2=alpha2, alpha2_err=_scalar_err(result, al2_pi),
                    f1=f1, f1_err=_scalar_err(result, f1_pi), f2=f2,
                    M0=M0, D0_m2ms=D0_FIXED, RN=rn,
                    model_name=MODEL_NAME,
                    g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
                    cost=float(result.cost), success=bool(result.success),
                ))

else:
    # ── Global fit: shared params over the specified scope ────────────────────
    scope_name, scope_cols = _group_scope()
    groups = data.groupby(scope_cols) if scope_cols else [("all", data)]

    for group_key, group_df in groups:
        (result, curves,
         per_tc1s, per_tc1e, per_tc2s, per_tc2e,
         per_al1s, per_al1e, per_al2s, per_al2e,
         per_f1s, per_f1e, per_f2s, M0s) = fit_group_global(group_df)

        key_str = "  ".join(str(k) for k in group_key) if isinstance(group_key, tuple) else str(group_key)
        print(f"[{key_str}]  cost={result.cost:.3e}")

        for (subj, roi, direction), sub_grp in group_df.groupby(["subj", "roi", "direction"]):
            entry = fit_store.setdefault((subj, roi, direction), {"tds": [], "fits": {}})
            for _td in sorted(float(t) for t in sub_grp.td_ms.unique()):
                if _td not in entry["fits"]:
                    entry["tds"].append(_td)
                    entry["fits"][_td] = {}
            entry["tds"].sort()

        for c, tc1, tc1e, tc2, tc2e, al1, al1e, al2, al2e, f1, f1e, f2 in zip(
            curves,
            per_tc1s, per_tc1e, per_tc2s, per_tc2e,
            per_al1s, per_al1e, per_al2s, per_al2e,
            per_f1s, per_f1e, per_f2s,
        ):
            subj, roi, direction, td, N = c["subj"], c["roi"], c["direction"], c["td"], c["N"]
            M0 = M0s[(subj, roi, direction, td, N)]
            fit_store[(subj, roi, direction)]["fits"][td][N] = dict(
                G=c["G"], y=c["y"], tc1=tc1, tc2=tc2, alpha1=al1, alpha2=al2,
                f1=f1, f2=f2, M0=M0, RN=c["rn"], S0=c["S0"],
            )
            rows.append(dict(
                subj=subj, roi=roi, direction=direction, td_ms=td, N=N,
                tc1_ms=tc1, tc1_err=tc1e, tc2_ms=tc2, tc2_err=tc2e,
                alpha1=al1, alpha1_err=al1e, alpha2=al2, alpha2_err=al2e,
                f1=f1, f1_err=f1e, f2=f2,
                M0=M0, D0_m2ms=D0_FIXED, RN=c["rn"],
                model_name=MODEL_NAME,
                g_column=G_COLUMN, g_correction_column=G_CORRECTION_COLUMN, y_column=Y_COLUMN,
                cost=float(result.cost), success=bool(result.success),
            ))

[BRAIN  AntCC  76.0]  cost=2.857e+04
[BRAIN  AntCC  90.0]  cost=2.119e+04
[BRAIN  AntCC  120.0]  cost=2.583e+04
[BRAIN  AntCC  143.4]  cost=1.100e+13
[BRAIN  AntCC  210.0]  cost=2.916e+04
[BRAIN  CentralCC  76.0]  cost=4.867e+04
[BRAIN  CentralCC  90.0]  cost=2.700e+04
[BRAIN  CentralCC  120.0]  cost=3.383e+04
[BRAIN  CentralCC  143.4]  cost=3.242e+04
[BRAIN  CentralCC  210.0]  cost=4.946e+04
[BRAIN  Left-Lateral-Ventricle  76.0]  cost=2.904e-04
[BRAIN  Left-Lateral-Ventricle  90.0]  cost=1.668e-04
[BRAIN  Left-Lateral-Ventricle  120.0]  cost=1.004e-03
[BRAIN  Left-Lateral-Ventricle  143.4]  cost=5.409e-04
[BRAIN  Left-Lateral-Ventricle  210.0]  cost=1.050e-03
[BRAIN  MidAntCC  76.0]  cost=3.025e+04
[BRAIN  MidAntCC  90.0]  cost=3.035e+04
[BRAIN  MidAntCC  120.0]  cost=4.100e+04
[BRAIN  MidAntCC  143.4]  cost=4.082e+04
[BRAIN  MidAntCC  210.0]  cost=3.067e+04
[BRAIN  MidPostCC  76.0]  cost=5.075e+04
[BRAIN  MidPostCC  90.0]  cost=3.721e+04
[BRAIN  MidPostCC  120.0]  cost=2.237e+04
[BRA

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE RESULTS TABLE
# Columns: subj, roi, direction, td_ms, N,
#          tc1_ms, tc1_err, tc2_ms, tc2_err,
#          alpha1, alpha1_err, alpha2, alpha2_err,
#          f1, f1_err, f2, M0, D0_m2ms, RN, cost, ...
# ══════════════════════════════════════════════════════════════════════════════

results_df = pd.DataFrame(rows)
results_df.to_excel(OUT_DIR / "fit_results.xlsx", index=False)
print(f"Saved {len(results_df)} rows → {OUT_DIR / 'fit_results.xlsx'}")
results_df

Saved 362 rows → ../../analysis/brains/ogse_experiments/lab/fit_rest-tort_CC-testb4/fit_results.xlsx


,subj,roi,direction,td_ms,N,tc1_ms,tc1_err,tc2_ms,tc2_err,alpha1,...,f2,M0,D0_m2ms,RN,model_name,g_column,g_correction_column,y_column,cost,success
0,BRAIN,AntCC,long,76.0,4,360.264260,0.000000,102.206409,3.751650e+10,1.000000e+00,...,1.000000,0.916891,3.200000e-12,37.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,28568.644136,True
1,BRAIN,AntCC,long,76.0,8,999.999687,0.000000,254.911961,0.000000e+00,1.000000e+00,...,1.000000,0.971912,3.200000e-12,37.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,28568.644136,True
2,BRAIN,AntCC,tra,76.0,4,999.999804,0.000000,779.280686,0.000000e+00,1.000000e+00,...,1.000000,0.906389,3.200000e-12,37.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,28568.644136,True
3,BRAIN,AntCC,tra,76.0,8,999.999670,0.000000,993.994579,1.360852e+10,1.000000e+00,...,1.000000,0.969486,3.200000e-12,37.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,28568.644136,True
4,BRAIN,AntCC,long,90.0,4,258.466108,0.000000,970.031973,0.000000e+00,9.999272e-01,...,0.048672,0.910154,3.200000e-12,32.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,21185.858209,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,MBBL,Right-Lateral-Ventricle,tra,143.4,8,0.194434,0.000039,25.290109,2.388249e+00,2.607897e-02,...,0.971933,1.000000,3.200000e-12,0.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,0.000498,True
358,MBBL,Right-Lateral-Ventricle,long,210.0,4,0.081953,0.000003,609.600280,0.000000e+00,1.512722e-26,...,0.964566,1.000000,3.200000e-12,0.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,0.001214,True
359,MBBL,Right-Lateral-Ventricle,long,210.0,8,0.209732,0.000009,29.874473,3.965030e-12,3.568835e-02,...,0.964566,1.000000,3.200000e-12,0.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,0.001214,True
360,MBBL,Right-Lateral-Ventricle,tra,210.0,4,0.073966,0.000003,134.847542,5.123982e+01,1.878953e-30,...,0.964566,1.000000,3.200000e-12,0.0,mixed-mixed,g_thorsten,grad_correction_factor,value_norm,0.001214,True


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT FITS — raw signal with Rician noise correction
# One figure per (subj, roi, dir): data points + fitted S(G) per (td, N)
# Top row:    S(G) = sqrt( (M0·(f1·S1+f2·S2))² + RN² )
# Bottom row: signal contrast  ΔS = S(N_hi) − S(N_lo)  vs G
#             restricted to min(max_G_N_hi, max_G_N_lo)
# ══════════════════════════════════════════════════════════════════════════════

G_plot = np.linspace(0, float(data.G_fit.max()), 300)
N_hi, N_lo = N_LIST[-1], N_LIST[0]

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes_grid = plt.subplots(2, n_tds, figsize=(4 * n_tds, 7), sharey="row", squeeze=False)
    axes      = axes_grid[0]
    diff_axes = axes_grid[1]

    for ax, dax, td in zip(axes, diff_axes, tds):
        fits_td = store["fits"][td]
        title_lines = [f"td = {td:.1f} ms"]
        y_hat_per_N = {}
        max_G_per_N = {}
        for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
            fit = fits_td.get(N)
            if fit is None:
                continue
            max_G_per_N[N] = float(fit["G"].max())
            ax.scatter(fit["G"], fit["y"], color=color, s=20, zorder=3, label=f"N={N} data")
            y_hat = _predict(td, G_plot, N,
                             fit["tc1"], fit["alpha1"], fit["tc2"], fit["alpha2"],
                             fit["f1"], fit["f2"], fit["M0"], fit["RN"])
            y_hat_per_N[N] = y_hat
            ax.plot(G_plot, y_hat, color=color, label=f"N={N} fit")
            title_lines.append(
                f"tc1={fit['tc1']:.2f} ms  tc2={fit['tc2']:.2f} ms  "
                f"α1={fit['alpha1']:.3f}  α2={fit['alpha2']:.3f}  "
                f"f1={fit['f1']:.3f}  RN={fit['RN']:.1f}  (N={N})"
            )
        ax.set_title("\n".join(title_lines), fontsize=6)
        ax.set_xlabel(G_LABEL)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
            G_diff = np.linspace(0, min(max_G_per_N[N_hi], max_G_per_N[N_lo]), 300)
            fit_hi, fit_lo = fits_td[N_hi], fits_td[N_lo]
            diff_hi = _predict(td, G_diff, N_hi, fit_hi["tc1"], fit_hi["alpha1"],
                               fit_hi["tc2"], fit_hi["alpha2"],
                               fit_hi["f1"], fit_hi["f2"], fit_hi["M0"], fit_hi["RN"])
            diff_lo = _predict(td, G_diff, N_lo, fit_lo["tc1"], fit_lo["alpha1"],
                               fit_lo["tc2"], fit_lo["alpha2"],
                               fit_lo["f1"], fit_lo["f2"], fit_lo["M0"], fit_lo["RN"])
            dax.plot(G_diff, diff_hi - diff_lo, color="k")
            dax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
        #     dax.axhline(fit_lo["RN"]/fit_lo["M0"], color="k", linewidth=0.7, linestyle="--", label = f"RN/M0 - N={N_lo}")
        #     dax.axhline(fit_hi["RN"]/fit_hi["M0"], color="k", linewidth=0.7, linestyle="--", label = f"RN/M0 - N={N_hi}")
        # dax.legend()
        dax.set_title(f"td = {td:.1f} ms  (N={N_hi} − N={N_lo})", fontsize=8)
        dax.set_xlabel(G_LABEL)
        dax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        dax.set_axisbelow(True)

    axes[0].set_ylabel(Y_LABEL)
    diff_axes[0].set_ylabel(f"ΔS  (N={N_hi} − N={N_lo})")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Fit plots saved.")

Fit plots saved.


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT parameters vs td  —  all subjects, grouped by ROI and direction
# One figure per (param, direction); one panel per ROI
# Colour = subject (lighter → darker within ROI palette), linestyle = N
# ══════════════════════════════════════════════════════════════════════════════

rois     = sorted(results_df.roi.unique())
subjects = sorted(results_df.subj.unique())
roi_colors   = _roi_color_map(rois)
_N_ls        = ["-", "--", "-.", ":"]
N_linestyles = {N: ls for N, ls in zip(N_LIST, _N_ls)}

for direction in DIRECTIONS:
    sub = results_df[results_df.direction == direction]
    for param, ylabel in [
        ("tc1_ms", "tc1 [ms]"), ("tc2_ms", "tc2 [ms]"),
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(1, len(rois), figsize=(4 * len(rois), 4), sharey=True)
        axes = np.atleast_1d(axes)

        for ax, roi in zip(axes, rois):
            base_color = roi_colors[roi]
            cmap = mcolors.LinearSegmentedColormap.from_list("", ["#cccccc", base_color])
            for i, subj in enumerate(subjects):
                shade  = cmap(0.3 + 0.7 * i / max(1, len(subjects) - 1))
                marker = DEFAULT_BRAIN_MARKERS[i % len(DEFAULT_BRAIN_MARKERS)]
                for N in N_LIST:
                    g = (
                        sub[(sub.roi == roi) & (sub.subj == subj) & (sub.N == N)]
                        .drop_duplicates("td_ms").sort_values("td_ms")
                    )
                    if g.empty:
                        continue
                    label = subj if N == N_LIST[0] else None
                    ax.scatter(g.td_ms, g[param], color=shade, marker=marker, s=60, zorder=3, label=label)
                    ax.plot(g.td_ms, g[param], color=shade, linewidth=0.8,
                            linestyle=N_linestyles[N])
            ax.set_title(roi, fontsize=9)
            ax.set_xlabel("td [ms]")
            ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)

        axes[0].set_ylabel(ylabel)
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, fontsize=7, loc="upper right", ncol=1)
        n_handles = [
            mlines.Line2D([], [], color="gray", linestyle=N_linestyles[N], label=f"N={N}")
            for N in N_LIST
        ]
        axes[-1].legend(handles=n_handles, fontsize=7, loc="lower right")
        fig.suptitle(f"{param} vs td — direction: {direction}", fontsize=10)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{direction}.png", dpi=120)
        plt.close(fig)

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT NORMALIZED FITS  —  M0·(f1·S1 + f2·S2) with no Rician correction
# Useful to compare model shape without amplitude effects.
# Data is corrected: sqrt(|y² − RN²|) / M0 → compared to f1·S1 + f2·S2
# Bottom row diff restricted to min(max_G_N_hi, max_G_N_lo)
# Horizontal dotted lines show RN/M0 per N
# ══════════════════════════════════════════════════════════════════════════════

for (subj, roi, direction), store in fit_store.items():
    tds   = store["tds"]
    n_tds = len(tds)

    fig, axes_grid = plt.subplots(2, n_tds, figsize=(4 * n_tds, 7), sharey="row", squeeze=False)
    axes      = axes_grid[0]
    diff_axes = axes_grid[1]

    for ax, dax, td in zip(axes, diff_axes, tds):
        fits_td = store["fits"][td]
        y_hat_per_N = {}
        max_G_per_N = {}
        N_color_map = {}
        for N, color in zip(N_LIST, ["C0", "C1", "C2", "C3"]):
            fit = fits_td.get(N)
            if fit is None:
                continue
            N_color_map[N] = color
            max_G_per_N[N] = float(fit["G"].max())
            M0 = fit["M0"] if fit["M0"] != 0.0 else 1.0
            RN = fit["RN"]
            y_corr = np.sqrt(np.abs(fit["y"]**2 - RN**2))*np.sign(fit["y"]**2 - RN**2) / M0
            ax.scatter(fit["G"], y_corr, color=color, s=20, zorder=3, label=f"N={N} data corr")
            with np.errstate(over="ignore", invalid="ignore"):
                s1 = M_ogse_mixed_offset(td, G_plot, N, td / N, fit["tc1"], fit["alpha1"], 1.0, D0_FIXED, 0, 0)
                s2 = M_ogse_mixed_offset(td, G_plot, N, td / N, fit["tc2"], fit["alpha2"], 1.0, D0_FIXED, 0, 0)
                sig_norm = fit["f1"] * s1 + fit["f2"] * s2
            y_hat_per_N[N] = sig_norm
            ax.plot(G_plot, sig_norm, color=color, label=f"N={N} model")
        ax.set_title(f"td = {td:.1f} ms", fontsize=8)
        ax.set_xlabel(G_LABEL)
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        ax.set_axisbelow(True)
        if N_hi in y_hat_per_N and N_lo in y_hat_per_N:
            G_diff = np.linspace(0, min(max_G_per_N[N_hi], max_G_per_N[N_lo]), 300)
            fit_hi, fit_lo = fits_td[N_hi], fits_td[N_lo]
            with np.errstate(over="ignore", invalid="ignore"):
                s1_hi = M_ogse_mixed_offset(td, G_diff, N_hi, td / N_hi, fit_hi["tc1"], fit_hi["alpha1"], 1.0, D0_FIXED, 0, 0)
                s2_hi = M_ogse_mixed_offset(td, G_diff, N_hi, td / N_hi, fit_hi["tc2"], fit_hi["alpha2"], 1.0, D0_FIXED, 0, 0)
                s1_lo = M_ogse_mixed_offset(td, G_diff, N_lo, td / N_lo, fit_lo["tc1"], fit_lo["alpha1"], 1.0, D0_FIXED, 0, 0)
                s2_lo = M_ogse_mixed_offset(td, G_diff, N_lo, td / N_lo, fit_lo["tc2"], fit_lo["alpha2"], 1.0, D0_FIXED, 0, 0)
            diff_hi = fit_hi["f1"] * s1_hi + fit_hi["f2"] * s2_hi
            diff_lo = fit_lo["f1"] * s1_lo + fit_lo["f2"] * s2_lo
            dax.plot(G_diff, diff_hi - diff_lo, color="k")
            dax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
            # for N_ref, fit_ref in [(N_hi, fit_hi), (N_lo, fit_lo)]:
            #     m0_ref = fit_ref["M0"] if fit_ref["M0"] != 0.0 else 1.0
            #     rn_ref = fit_ref["RN"]
                # if rn_ref > 0:
                #     dax.axhline(rn_ref / m0_ref, color=N_color_map.get(N_ref, "gray"),
                #                 linewidth=0.7, linestyle=":", label=f"RN/M0 (N={N_ref})")
            dax.legend(fontsize=6)
        dax.set_title(f"td = {td:.1f} ms  (N={N_hi} − N={N_lo})", fontsize=8)
        dax.set_xlabel(G_LABEL)
        dax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
        dax.set_axisbelow(True)

    axes[0].set_ylabel(f"{Y_COLUMN} / M0 [a.u.]")
    diff_axes[0].set_ylabel(f"ΔS  (N={N_hi} − N={N_lo})")
    axes[0].legend(fontsize=6)
    fig.suptitle(f"{subj}  |  {roi}  |  {direction}  |  f1·S1+f2·S2  (clean model)", fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"fit_norm_{subj}_{roi}_{direction}.png", dpi=120)
    plt.close(fig)

print("Normalized model plots saved.")

/tmp/ipykernel_3215111/1036252580.py:60: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  dax.legend(fontsize=6)


Normalized model plots saved.


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT tc1, tc2, α1, α2, f1 and M0 vs td  —  per subject
# Rows = ROIs, columns = directions, colour/linestyle = N
# Shaded band = ±1σ from the Jacobian covariance
# ══════════════════════════════════════════════════════════════════════════════

subjects_plot = sorted(results_df.subj.unique())
rois_plot     = sorted(results_df.roi.unique())
Ns_plot       = sorted(results_df.N.unique())
dirs_plot     = [d for d in ["long", "tra"] if d in results_df.direction.unique()]
_N_ls_p       = ["-", "--", "-.", ":"]
N_ls          = {N: _N_ls_p[i % len(_N_ls_p)] for i, N in enumerate(Ns_plot)}

for subj in subjects_plot:
    for param, ylabel in [
        ("tc1_ms", "tc1 [ms]"), ("tc2_ms", "tc2 [ms]"),
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"), ("M0", "M0 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False,
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                s = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = s[s.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x    = g.td_ms.to_numpy(dtype=float)
                        yv   = g[param].to_numpy(dtype=float)
                        yerr = g[err_col].to_numpy(dtype=float)
                        ok   = np.isfinite(x) & np.isfinite(yv) & np.isfinite(yerr)
                        if np.any(ok):
                            ax.fill_between(x[ok], yv[ok] - yerr[ok], yv[ok] + yerr[ok],
                                            color=line.get_color(), alpha=0.18, linewidth=0)
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel}", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_{subj}.png", dpi=120)
        plt.close(fig)

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT tc1, tc2, α1, α2, f1 and M0 vs td  —  per subject, log-scale y-axis
# Same layout as previous cell; useful to spot order-of-magnitude variation
# ══════════════════════════════════════════════════════════════════════════════

for subj in subjects_plot:
    for param, ylabel in [
        ("tc1_ms", "tc1 [ms]"), ("tc2_ms", "tc2 [ms]"),
        ("alpha1", "α1 [a.u.]"), ("alpha2", "α2 [a.u.]"),
        ("f1", "f1 [a.u.]"), ("M0", "M0 [a.u.]"),
    ]:
        if param not in results_df.columns:
            continue
        fig, axes = plt.subplots(
            len(rois_plot), len(dirs_plot),
            figsize=(4 * len(dirs_plot), 3 * len(rois_plot)),
            sharey=False, squeeze=False,
        )
        for i_roi, roi in enumerate(rois_plot):
            for i_dir, direction in enumerate(dirs_plot):
                ax = axes[i_roi, i_dir]
                s = results_df[
                    (results_df.subj == subj) &
                    (results_df.roi == roi) &
                    (results_df.direction == direction)
                ]
                for N in Ns_plot:
                    g = s[s.N == N].sort_values("td_ms")
                    if g.empty:
                        continue
                    line, = ax.plot(g.td_ms, g[param], linestyle=N_ls[N],
                                    marker="o", ms=4, linewidth=0.8, label=f"N={N}")
                    err_col = f"{param}_err"
                    if err_col in g.columns:
                        x     = g.td_ms.to_numpy(dtype=float)
                        yv    = g[param].to_numpy(dtype=float)
                        yerr  = g[err_col].to_numpy(dtype=float)
                        lower = np.maximum(yv - yerr, np.finfo(float).tiny)
                        upper = yv + yerr
                        ok    = np.isfinite(x) & np.isfinite(yv) & np.isfinite(yerr) & (yv > 0) & (upper > 0)
                        if np.any(ok):
                            ax.fill_between(x[ok], lower[ok], upper[ok],
                                            color=line.get_color(), alpha=0.18, linewidth=0)
                ax.set_yscale("log")
                if i_roi == 0:
                    ax.set_title(direction, fontsize=9)
                if i_dir == 0:
                    ax.set_ylabel(f"{roi}\n{ylabel} (log)", fontsize=8)
                ax.set_xlabel("td [ms]")
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                if i_roi == 0 and i_dir == len(dirs_plot) - 1:
                    ax.legend(fontsize=6, title="N", title_fontsize=6)
        fig.suptitle(f"{subj} — {param} vs td  (log scale)", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"{param}_vs_td_log_{subj}.png", dpi=120)
        plt.close(fig)

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT SIGNAL CONTRAST  ΔS = S(N_hi) − S(N_lo)  vs G, Ld, Lcf, lcf
# Per subject: rows = ROIs, columns = directions, colour = td
# Contrast computed over the common G range: min(max_G_N_hi, max_G_N_lo)
# "_raw"  uses the full model:  sqrt( (M0·(f1·S1+f2·S2))² + RN² )
# "_corr" uses the clean model: M0·(f1·S1+f2·S2)  (no Rician term)
# ══════════════════════════════════════════════════════════════════════════════

td_cmap    = plt.cm.viridis
N_hi, N_lo = N_LIST[-1], N_LIST[0]
G_plot     = np.linspace(0, float(data.G_fit.max()), 300)
PEAK_GAMMA = 267.5221900   # rad / (ms · mT), proton gyromagnetic ratio
D0_PLOT    = D0_FIXED

def _x_for_G(G_arr, td_ms, xvar):
    if xvar == "G":
        return G_arr.copy()
    D0, gamma = float(D0_PLOT), float(PEAK_GAMMA)
    l_d = np.sqrt(D0 * float(td_ms))
    l_G = np.full_like(G_arr, np.nan)
    valid = G_arr > 0
    l_G[valid] = (D0 / (gamma * G_arr[valid])) ** (1.0 / 3.0)
    Ld  = l_d / l_G
    Lcf = 1.5 ** 0.25 / np.sqrt(Ld)
    lcf = Lcf * l_G * 1e6
    if xvar == "Ld":  return Ld
    if xvar == "Lcf_a": return Lcf
    if xvar == "lcf": return lcf
    raise ValueError(xvar)

X_AXES = [
    ("G",   G_LABEL),
    ("Ld",  "Ld (dimensionless)"),
    ("Lcf_a", "Lcf (dimensionless)"),
    ("lcf", "lcf [µm]"),
]

X_LIMITS = {
    "Lcf_a": (0.5, 1.4),
    "lcf": (0, 35),
}

# ── precompute contrasts ───────────────────────────────────────────────────────
raw_contrast_store  = {}
corr_contrast_store = {}
contrast_G_store    = {}   # G_common per (subj, roi, direction, td)
free_contrast_store = {}   # free (unrestricted) model contrast, same D0 as other models

for (subj, roi, direction), store in fit_store.items():
    for td in store["tds"]:
        fits_td = store["fits"][td]
        max_G_per_N = {
            N: float(fits_td[N]["G"].max())
            for N in N_LIST if fits_td.get(N) is not None
        }
        if N_hi not in max_G_per_N or N_lo not in max_G_per_N:
            continue
        G_common = np.linspace(0, min(max_G_per_N[N_hi], max_G_per_N[N_lo]), 300)
        key = (subj, roi, direction, td)
        raw_per_N  = {}
        corr_per_N = {}
        for N in N_LIST:
            fit = fits_td.get(N)
            if fit is None:
                continue
            rn = fit.get("RN", 0.0)
            with np.errstate(over="ignore", invalid="ignore"):
                s1 = M_ogse_mixed_offset(td, G_common, N, td / N, fit["tc1"], fit["alpha1"], 1, D0_FIXED, 0, 0)
                s2 = M_ogse_mixed_offset(td, G_common, N, td / N, fit["tc2"], fit["alpha2"], 1, D0_FIXED, 0, 0)
                bimodal  = fit["f1"] * s1 + fit["f2"] * s2
                raw_per_N[N]  = np.sqrt(bimodal**2 + (rn / fit["M0"])**2) if rn != 0.0 else bimodal
                corr_per_N[N] = bimodal
        if N_hi in raw_per_N and N_lo in raw_per_N:
            raw_contrast_store[key]  = raw_per_N[N_hi]  - raw_per_N[N_lo]
            contrast_G_store[key]    = G_common
        if N_hi in corr_per_N and N_lo in corr_per_N:
            corr_contrast_store[key] = corr_per_N[N_hi] - corr_per_N[N_lo]
        with np.errstate(over="ignore", invalid="ignore"):
            sig_hi = M_ogse_free(td, G_common, N_hi, td / N_hi, 1, D0_FIXED)
            sig_lo = M_ogse_free(td, G_common, N_lo, td / N_lo, 1, D0_FIXED)
        free_contrast_store[key] = sig_hi - sig_lo


def _plot_contrast(cstore, xvar, xlabel, subj, suffix, title_tag, free_store=None):
    rois_subj = sorted(set(k[1] for k in cstore if k[0] == subj))
    dirs_subj = [d for d in DIRECTIONS if any(k[2] == d for k in cstore if k[0] == subj)]
    n_rois, n_dirs = len(rois_subj), len(dirs_subj)
    if n_rois == 0 or n_dirs == 0:
        return
    all_tds_subj = sorted(set(k[3] for k in cstore if k[0] == subj))
    colors_leg   = [td_cmap(i / max(1, len(all_tds_subj) - 1)) for i in range(len(all_tds_subj))]
    leg_handles  = [plt.Line2D([0], [0], color=c, linewidth=1.5) for c in colors_leg]
    leg_labels   = [f"td={td:.0f} ms" for td in all_tds_subj]
    td_color_map = {td: c for td, c in zip(all_tds_subj, colors_leg)}
    if free_store is not None:
        leg_handles.append(plt.Line2D([0], [0], color="lightgrey", linestyle=":", linewidth=1.5))
        leg_labels.append("free")
    fig, axes = plt.subplots(n_rois, n_dirs, figsize=(4 * n_dirs, 3 * n_rois), squeeze=False, sharey="row")
    for i_roi, roi in enumerate(rois_subj):
        for i_dir, direction in enumerate(dirs_subj):
            ax = axes[i_roi, i_dir]
            tds_here = sorted(k[3] for k in cstore
                              if k[0] == subj and k[1] == roi and k[2] == direction)
            if not tds_here:
                ax.set_visible(False)
                continue
            for td in tds_here:
                key = (subj, roi, direction, td)
                G_arr = contrast_G_store.get(key)
                if G_arr is None:
                    continue
                ax.plot(_x_for_G(G_arr, td, xvar), cstore[key],
                        color=td_color_map[td], linewidth=1.2)
                if free_store is not None and key in free_store:
                    ax.plot(_x_for_G(G_arr, td, xvar), free_store[key],
                            color="lightgrey", linestyle=":", linewidth=1.2)
            ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
            if i_roi == 0:
                ax.set_title(direction, fontsize=9)
            if i_dir == 0:
                ax.set_ylabel(f"{roi}\nΔS (N={N_hi}−N={N_lo})", fontsize=7)
            ax.set_xlabel(xlabel, fontsize=7)
            ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)
            ax.set_axisbelow(True)
            ax.tick_params(labelsize=6)
            if xvar in X_LIMITS:
                ax.set_xlim(*X_LIMITS[xvar])
            ax.tick_params(axis="y", labelleft=True)
    fig.suptitle(f"{subj} — contrast ΔS (N={N_hi}−N={N_lo}) vs {xvar}  [{title_tag}]", fontsize=10)
    fig.legend(leg_handles, leg_labels,
               loc="upper center", ncol=len(leg_labels), fontsize=7, frameon=False,
               bbox_to_anchor=(0.5, 0.96), bbox_transform=fig.transFigure)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(OUT_DIR / f"contrast_vs_{xvar}_{subj}_{suffix}.png", dpi=120, bbox_inches="tight")
    plt.close(fig)


all_subjs = sorted(set(k[0] for k in raw_contrast_store))
for xvar, xlabel in X_AXES:
    for subj in all_subjs:
        _plot_contrast(raw_contrast_store,  xvar, xlabel, subj, "raw",  "fit w/ RN")
        _plot_contrast(corr_contrast_store, xvar, xlabel, subj, "corr", "clean model", free_store=free_contrast_store)

print("Per-subject contrast figures saved (_raw and _corr).")

/tmp/ipykernel_3215111/3456633088.py:71: RuntimeWarning: divide by zero encountered in scalar divide
  raw_per_N[N]  = np.sqrt(bimodal**2 + (rn / fit["M0"])**2) if rn != 0.0 else bimodal
/tmp/ipykernel_3215111/3456633088.py:74: RuntimeWarning: invalid value encountered in subtract
  raw_contrast_store[key]  = raw_per_N[N_hi]  - raw_per_N[N_lo]


Per-subject contrast figures saved (_raw and _corr).


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTRAST CURVE FITTING — CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# Fitting target: corr_contrast_store[(subj,roi,dir,td)]  — model contrast (M0=1)
#                 evaluated on contrast_G_store[(subj,roi,dir,td)]  (dense G_common)
# M0 = 1 always.  D0 always fixed.  Only tc and alpha are free or pinned.

CF_OUT_SUBDIR = "contrast_fits_corr"  # sub-folder inside OUT_DIR

# ── Free model ────────────────────────────────────────────────────────────────
CF_FREE_D0_FIXED   = D0_FIXED   # m²/ms

# ── Rest model ────────────────────────────────────────────────────────────────
CF_REST_TC_INIT   = 2.0         # ms
CF_REST_TC_FIXED  = None        # None → fitted; float → pinned
CF_REST_TC_BOUNDS = (0.05, 1000.0)
CF_REST_D0_FIXED  = D0_FIXED

# ── Tort model ────────────────────────────────────────────────────────────────
CF_TORT_ALPHA_INIT       = 0.5
CF_TORT_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_TORT_ALPHA_BOUNDS     = (0.0, 1.0)
CF_TORT_ALPHA_MASTER_COL = "alpha_macro"
CF_TORT_D0_FIXED         = D0_FIXED

# ── Mixed model ───────────────────────────────────────────────────────────────
CF_MIXED_TC_INIT          = 2.0
CF_MIXED_TC_FIXED         = None
CF_MIXED_TC_BOUNDS        = (0.05, 1000.0)
CF_MIXED_ALPHA_INIT       = 0.5
CF_MIXED_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_MIXED_ALPHA_BOUNDS     = (0.0, 1.0)
CF_MIXED_ALPHA_MASTER_COL = "alpha_macro"
CF_MIXED_D0_FIXED         = D0_FIXED

In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT CONTRAST CURVES — 4 SINGLE-COMPONENT MODELS
# Fitting target: corr_contrast_store (M0-normalised, noise-free model contrast)
# M0 = 1 always.  D0 fixed per model.  tc fitted in log-space; alpha linear.
# ══════════════════════════════════════════════════════════════════════════════

from portable_helpers import (
    OGSE_contrast_vs_g_free, OGSE_contrast_vs_g_rest,
    OGSE_contrast_vs_g_tort, OGSE_contrast_vs_g_mixed,
)

(OUT_DIR / CF_OUT_SUBDIR).mkdir(parents=True, exist_ok=True)

contrast_fit_store = {}
cf_rows = []

for key in sorted(corr_contrast_store):
    subj, roi, direction, td = key
    G_fit = contrast_G_store[key]
    C_fit = corr_contrast_store[key]

    # ── circle markers: corr contrast values at actual data G points ──────────
    fits_td   = fit_store.get((subj, roi, direction), {}).get("fits", {}).get(td, {})
    G_hi_data = fits_td[N_hi]["G"] if N_hi in fits_td else np.array([])
    G_lo_data = fits_td[N_lo]["G"] if N_lo in fits_td else np.array([])
    C_at_Ghi  = np.interp(G_hi_data, G_fit, C_fit)
    C_at_Glo  = np.interp(G_lo_data, G_fit, C_fit)

    entry = {
        "G_common": G_fit, "C_corr": C_fit,
        "G_hi": G_hi_data, "C_hi": C_at_Ghi,
        "G_lo": G_lo_data, "C_lo": C_at_Glo,
    }

    # ── free model (no free physics params) ───────────────────────────────────
    _D0_f   = CF_FREE_D0_FIXED
    _C_free = OGSE_contrast_vs_g_free(td, G_fit, G_fit, N_hi, N_lo, 1.0, _D0_f)
    entry["free"] = {
        "D0": _D0_f, "C_curve": _C_free,
        "cost": float(0.5 * np.sum((_C_free - C_fit) ** 2)),
    }

    # ── rest model ────────────────────────────────────────────────────────────
    _D0_r = CF_REST_D0_FIXED
    if CF_REST_TC_FIXED is not None:
        _tc_r = float(CF_REST_TC_FIXED)
    else:
        def _res_rest(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_r):
            r = OGSE_contrast_vs_g_rest(_td, _G, _G, N_hi, N_lo, np.exp(p[0]), 1.0, _D0) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_rest = least_squares(
            _res_rest, [np.log(CF_REST_TC_INIT)],
            bounds=([np.log(CF_REST_TC_BOUNDS[0])], [np.log(CF_REST_TC_BOUNDS[1])]),
            method="trf", max_nfev=10000,
        )
        _tc_r = float(np.exp(_r_rest.x[0]))
    _C_rest = OGSE_contrast_vs_g_rest(td, G_fit, G_fit, N_hi, N_lo, _tc_r, 1.0, _D0_r)
    entry["rest"] = {
        "tc": _tc_r, "D0": _D0_r, "C_curve": _C_rest,
        "cost": float(0.5 * np.sum((_C_rest - C_fit) ** 2)),
    }

    # ── tort model ────────────────────────────────────────────────────────────
    _D0_t = CF_TORT_D0_FIXED
    if CF_TORT_ALPHA_FIXED == "master":
        _al_t = _get_master_value(subj, roi, direction, CF_TORT_ALPHA_MASTER_COL)
    elif CF_TORT_ALPHA_FIXED is not None:
        _al_t = float(CF_TORT_ALPHA_FIXED)
    else:
        def _res_tort(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_t):
            r = OGSE_contrast_vs_g_tort(_td, _G, _G, N_hi, N_lo, p[0], 1.0, _D0) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_tort = least_squares(
            _res_tort, [CF_TORT_ALPHA_INIT],
            bounds=([CF_TORT_ALPHA_BOUNDS[0]], [CF_TORT_ALPHA_BOUNDS[1]]),
            method="trf", max_nfev=10000,
        )
        _al_t = float(_r_tort.x[0])
    _C_tort = OGSE_contrast_vs_g_tort(td, G_fit, G_fit, N_hi, N_lo, _al_t, 1.0, _D0_t)
    entry["tort"] = {
        "alpha": _al_t, "D0": _D0_t, "C_curve": _C_tort,
        "cost": float(0.5 * np.sum((_C_tort - C_fit) ** 2)),
    }

    # ── mixed model ───────────────────────────────────────────────────────────
    _D0_m       = CF_MIXED_D0_FIXED
    _tc_m_fixed = float(CF_MIXED_TC_FIXED) if CF_MIXED_TC_FIXED is not None else None
    if CF_MIXED_ALPHA_FIXED == "master":
        _al_m_fixed = _get_master_value(subj, roi, direction, CF_MIXED_ALPHA_MASTER_COL)
    elif CF_MIXED_ALPHA_FIXED is not None:
        _al_m_fixed = float(CF_MIXED_ALPHA_FIXED)
    else:
        _al_m_fixed = None
    _x0_m, _lo_m, _hi_m = [], [], []
    if _tc_m_fixed is None:
        _x0_m.append(np.log(CF_MIXED_TC_INIT))
        _lo_m.append(np.log(CF_MIXED_TC_BOUNDS[0])); _hi_m.append(np.log(CF_MIXED_TC_BOUNDS[1]))
    if _al_m_fixed is None:
        _x0_m.append(CF_MIXED_ALPHA_INIT)
        _lo_m.append(CF_MIXED_ALPHA_BOUNDS[0]); _hi_m.append(CF_MIXED_ALPHA_BOUNDS[1])
    if _x0_m:
        def _res_mixed(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_m,
                       _tcf=_tc_m_fixed, _alf=_al_m_fixed):
            _i  = 0
            _tc = np.exp(p[_i]) if _tcf is None else _tcf; _i += int(_tcf is None)
            _al = p[_i]          if _alf is None else _alf
            r   = OGSE_contrast_vs_g_mixed(_td, _G, _G, N_hi, N_lo, _tc, _al, 1.0, _D0) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_mixed = least_squares(
            _res_mixed, _x0_m, bounds=(_lo_m, _hi_m), method="trf", max_nfev=10000,
        )
        _i    = 0
        _tc_m = float(np.exp(_r_mixed.x[_i])) if _tc_m_fixed is None else _tc_m_fixed
        _i   += int(_tc_m_fixed is None)
        _al_m = float(_r_mixed.x[_i])           if _al_m_fixed is None else _al_m_fixed
    else:
        _tc_m = _tc_m_fixed if _tc_m_fixed is not None else CF_MIXED_TC_INIT
        _al_m = _al_m_fixed if _al_m_fixed is not None else CF_MIXED_ALPHA_INIT
    _C_mixed = OGSE_contrast_vs_g_mixed(td, G_fit, G_fit, N_hi, N_lo, _tc_m, _al_m, 1.0, _D0_m)
    entry["mixed"] = {
        "tc": _tc_m, "alpha": _al_m, "D0": _D0_m, "C_curve": _C_mixed,
        "cost": float(0.5 * np.sum((_C_mixed - C_fit) ** 2)),
    }

    contrast_fit_store[key] = entry

    print(
        f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  "
        f"rest tc={entry['rest']['tc']:.3f} ms  "
        f"tort α={entry['tort']['alpha']:.3f}  "
        f"mixed tc={entry['mixed']['tc']:.3f} ms α={entry['mixed']['alpha']:.3f}"
    )
    cf_rows.append(dict(
        subj=subj, roi=roi, direction=direction, td_ms=td,
        free_cost   =entry["free"]["cost"],
        rest_tc_ms  =entry["rest"]["tc"],    rest_cost  =entry["rest"]["cost"],
        tort_alpha  =entry["tort"]["alpha"], tort_cost  =entry["tort"]["cost"],
        mixed_tc_ms =entry["mixed"]["tc"],   mixed_alpha=entry["mixed"]["alpha"],
        mixed_cost  =entry["mixed"]["cost"],
    ))

cf_df = pd.DataFrame(cf_rows)
_cf_xlsx = OUT_DIR / CF_OUT_SUBDIR / "contrast_fit_results.xlsx"
cf_df.to_excel(_cf_xlsx, index=False)
print(f"\nSaved {len(cf_df)} rows → {_cf_xlsx}")
cf_df

BRAIN   AntCC                      long  td=   76 ms  rest tc=728.186 ms  tort α=1.000  mixed tc=513.765 ms α=0.957
BRAIN   AntCC                      long  td=   90 ms  rest tc=557.775 ms  tort α=1.000  mixed tc=344.891 ms α=0.990
BRAIN   AntCC                      long  td=  120 ms  rest tc=10.367 ms  tort α=0.196  mixed tc=10.367 ms α=0.000
BRAIN   AntCC                      long  td=  143 ms  rest tc=11.926 ms  tort α=1.000  mixed tc=11.926 ms α=0.000
BRAIN   AntCC                      long  td=  210 ms  rest tc=999.296 ms  tort α=1.000  mixed tc=683.533 ms α=0.979
BRAIN   AntCC                      tra  td=   76 ms  rest tc=764.730 ms  tort α=1.000  mixed tc=556.512 ms α=0.961
BRAIN   AntCC                      tra  td=   90 ms  rest tc=562.716 ms  tort α=1.000  mixed tc=503.815 ms α=0.985
BRAIN   AntCC                      tra  td=  120 ms  rest tc=27.434 ms  tort α=0.964  mixed tc=27.434 ms α=0.000
BRAIN   AntCC                      tra  td=  143 ms  rest tc=33.539 ms  tort α=0.

/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:101: RuntimeWarning: overflow encountered in exp
  - np.exp(x / tc) / 2
/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:117: RuntimeWarning: invalid value encountered in multiply
  phi_cross = 2 * tc**2 * inner_cross * bSE**2 / (np.exp(x / tc) + 1)
/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:117: RuntimeWarning: overflow encountered in exp
  phi_cross = 2 * tc**2 * inner_cross * bSE**2 / (np.exp(x / tc) + 1)
/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:117: RuntimeWarning: invalid value encountered in divide
  phi_cross = 2 * tc**2 * inner_cross * bSE**2 / (np.exp(x / tc) + 1)


BRAIN   Left-Lateral-Ventricle     tra  td=   90 ms  rest tc=34.519 ms  tort α=0.917  mixed tc=34.519 ms α=0.000
BRAIN   Left-Lateral-Ventricle     tra  td=  120 ms  rest tc=60.445 ms  tort α=0.942  mixed tc=3.065 ms α=0.925
BRAIN   Left-Lateral-Ventricle     tra  td=  143 ms  rest tc=230.757 ms  tort α=0.960  mixed tc=0.051 ms α=0.960
BRAIN   Left-Lateral-Ventricle     tra  td=  210 ms  rest tc=106.397 ms  tort α=1.000  mixed tc=106.396 ms α=0.000
BRAIN   MidAntCC                   long  td=   76 ms  rest tc=549.053 ms  tort α=1.000  mixed tc=500.100 ms α=0.955
BRAIN   MidAntCC                   long  td=   90 ms  rest tc=549.226 ms  tort α=1.000  mixed tc=385.876 ms α=0.980
BRAIN   MidAntCC                   long  td=  120 ms  rest tc=739.439 ms  tort α=1.000  mixed tc=578.805 ms α=0.977
BRAIN   MidAntCC                   long  td=  143 ms  rest tc=782.435 ms  tort α=1.000  mixed tc=561.486 ms α=0.977
BRAIN   MidAntCC                   long  td=  210 ms  rest tc=999.941 ms  tort α=1.

,subj,roi,direction,td_ms,free_cost,rest_tc_ms,rest_cost,tort_alpha,tort_cost,mixed_tc_ms,mixed_alpha,mixed_cost
0,BRAIN,AntCC,long,76.0,1.628890e-10,728.186222,2.448023e-07,0.999956,3.945195e-09,513.765409,9.568251e-01,9.399708e-10
1,BRAIN,AntCC,long,90.0,2.458440e-10,557.775113,1.349098e-06,0.999976,1.822721e-09,344.891373,9.896025e-01,1.897969e-09
2,BRAIN,AntCC,long,120.0,2.056440e+00,10.367153,7.763390e-02,0.195887,1.118073e+00,10.367156,1.081054e-15,7.763390e-02
3,BRAIN,AntCC,long,143.4,1.806000e+00,11.926127,1.963054e-01,1.000000,1.806000e+00,11.926206,5.034583e-15,1.963054e-01
4,BRAIN,AntCC,long,210.0,4.618498e-14,999.296457,1.387233e-06,0.999975,3.018666e-09,683.533347,9.794679e-01,2.519345e-09
...,...,...,...,...,...,...,...,...,...,...,...,...
175,MBBL,Right-Lateral-Ventricle,long,210.0,1.470309e-02,109.597003,1.047569e-02,1.000000,1.470309e-02,109.590632,8.195016e-09,1.047569e-02
176,MBBL,Right-Lateral-Ventricle,tra,90.0,2.340465e-02,38.272331,8.299532e-04,0.943816,1.609917e-02,38.272294,2.957266e-10,8.299532e-04
177,MBBL,Right-Lateral-Ventricle,tra,120.0,1.496728e-02,54.084878,6.350068e-04,0.965823,8.927213e-03,9.611773,8.899878e-01,4.445536e-05
178,MBBL,Right-Lateral-Ventricle,tra,143.4,1.684946e-03,118.205089,2.639109e-04,0.988718,8.132806e-04,9.862915,9.647341e-01,9.387743e-05


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT CONTRAST FITS  —  per (subj, roi, td)
# One figure per (subj, roi, td); columns = directions
# Data: corr_contrast values at N_hi G points (filled ●) and N_lo G points (empty ○)
# Fits: free (C0), rest (C1), tort (C2), mixed (C3)
# ══════════════════════════════════════════════════════════════════════════════

_cf_subjs = sorted(set(k[0] for k in contrast_fit_store))
_cf_rois  = sorted(set(k[1] for k in contrast_fit_store))
_cf_tds   = sorted(set(k[3] for k in contrast_fit_store))

for subj in _cf_subjs:
    for roi in _cf_rois:
        for td in _cf_tds:
            dirs_here = [d for d in DIRECTIONS if (subj, roi, d, td) in contrast_fit_store]
            if not dirs_here:
                continue
            fig, axes = plt.subplots(
                1, len(dirs_here),
                figsize=(4.5 * len(dirs_here), 4),
                sharey=True, squeeze=False,
            )
            axes = axes[0]

            for ax, direction in zip(axes, dirs_here):
                entry = contrast_fit_store[(subj, roi, direction, td)]
                G_c   = entry["G_common"]

                # ── data markers (values of the corr contrast at actual G points) ──
                ax.scatter(entry["G_hi"], entry["C_hi"],
                           color="k", s=20, zorder=5, label=f"N={N_hi} pts")
                ax.scatter(entry["G_lo"], entry["C_lo"],
                           facecolors="none", edgecolors="k", s=60, zorder=5,
                           label=f"N={N_lo} pts")

                # ── model fit lines ───────────────────────────────────────────
                ax.plot(G_c, entry["free"]["C_curve"],  color="C0", lw=1.5, label="free")
                ax.plot(G_c, entry["rest"]["C_curve"],  color="C1", lw=1.5,
                        label=f"rest  tc={entry['rest']['tc']:.2f} ms")
                ax.plot(G_c, entry["tort"]["C_curve"],  color="C2", lw=1.5,
                        label=f"tort  α={entry['tort']['alpha']:.3f}")
                ax.plot(G_c, entry["mixed"]["C_curve"], color="C3", lw=1.5,
                        label=f"mixed  tc={entry['mixed']['tc']:.2f} ms, α={entry['mixed']['alpha']:.3f}")

                ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
                ax.set_title(direction, fontsize=9)
                ax.set_xlabel(G_LABEL)
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                ax.legend(fontsize=6, loc="best")

            axes[0].set_ylabel(f"ΔS (N={N_hi}−N={N_lo}) [a.u.]")
            fig.suptitle(
                f"{subj}  |  {roi}  |  td={td:.0f} ms  —  contrast fits", fontsize=10
            )
            fig.tight_layout()
            fig.savefig(
                OUT_DIR / CF_OUT_SUBDIR / f"contrast_fit_{subj}_{roi}_{td:.0f}.png",
                dpi=120,
            )
            plt.close(fig)

print("Contrast fit plots saved.")

Contrast fit plots saved.


In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTRAST CURVE FITTING (RAW) — CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
# Fitting target: raw_contrast_store[(subj,roi,dir,td)]
#                 sqrt( (f1·S1+f2·S2)² + (RN/M0_N)² ) at N_hi and N_lo, then subtracted.
# RN/M0 uses each N's own M0 = sqrt(S0²−RN²), matching cell 11.
# M0 = 1 in model calls (M0 absorbed into rn_hi / rn_lo).  D0 always fixed.
# ══════════════════════════════════════════════════════════════════════════════

CF_RAW_OUT_SUBDIR = "contrast_fits_raw"  # sub-folder inside OUT_DIR

# ── Free model ────────────────────────────────────────────────────────────────
CF_RAW_FREE_D0_FIXED   = D0_FIXED   # m²/ms

# ── Rest model ────────────────────────────────────────────────────────────────
CF_RAW_REST_TC_INIT   = 2.0         # ms
CF_RAW_REST_TC_FIXED  = None        # None → fitted; float → pinned
CF_RAW_REST_TC_BOUNDS = (0.05, 1000.0)
CF_RAW_REST_D0_FIXED  = D0_FIXED

# ── Tort model ────────────────────────────────────────────────────────────────
CF_RAW_TORT_ALPHA_INIT       = 0.5
CF_RAW_TORT_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_RAW_TORT_ALPHA_BOUNDS     = (0.0, 1.0)
CF_RAW_TORT_ALPHA_MASTER_COL = "alpha_macro"
CF_RAW_TORT_D0_FIXED         = D0_FIXED

# ── Mixed model ───────────────────────────────────────────────────────────────
CF_RAW_MIXED_TC_INIT          = 2.0
CF_RAW_MIXED_TC_FIXED         = None
CF_RAW_MIXED_TC_BOUNDS        = (0.05, 1000.0)
CF_RAW_MIXED_ALPHA_INIT       = 0.5
CF_RAW_MIXED_ALPHA_FIXED      = None     # None → fitted; float → pinned; "master" → from master table
CF_RAW_MIXED_ALPHA_BOUNDS     = (0.0, 1.0)
CF_RAW_MIXED_ALPHA_MASTER_COL = "alpha_macro"
CF_RAW_MIXED_D0_FIXED         = D0_FIXED

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# FIT CONTRAST CURVES (RAW) — 4 SINGLE-COMPONENT MODELS
# Fitting target: raw_contrast_store  (includes Rician noise term per N)
# Model:  sqrt(S_model(N_hi)² + (RN/M0_hi)²) − sqrt(S_model(N_lo)² + (RN/M0_lo)²)
# M0 = 1 in signal calls; RN normalised by each N's own M0 = sqrt(S0²−RN²).
# D0 fixed per model.  tc fitted in log-space; alpha linear.
# ══════════════════════════════════════════════════════════════════════════════

from portable_helpers import (
    M_ogse_free, M_ogse_rest, M_ogse_tort, M_ogse_mixed,
)

(OUT_DIR / CF_RAW_OUT_SUBDIR).mkdir(parents=True, exist_ok=True)

# ── raw-contrast wrappers: sqrt(S_hi² + rn_hi²) − sqrt(S_lo² + rn_lo²) ──────

def _Craw_free(td, G, N_hi, N_lo, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_free(td, G, N_hi, td / N_hi, M0, D0)
    S_lo = M_ogse_free(td, G, N_lo, td / N_lo, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

def _Craw_rest(td, G, N_hi, N_lo, tc, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_rest(td, G, N_hi, td / N_hi, tc, M0, D0)
    S_lo = M_ogse_rest(td, G, N_lo, td / N_lo, tc, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

def _Craw_tort(td, G, N_hi, N_lo, alpha, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_tort(td, G, N_hi, td / N_hi, alpha, M0, D0)
    S_lo = M_ogse_tort(td, G, N_lo, td / N_lo, alpha, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

def _Craw_mixed(td, G, N_hi, N_lo, tc, alpha, M0, D0, rn_hi, rn_lo):
    S_hi = M_ogse_mixed(td, G, N_hi, td / N_hi, tc, alpha, M0, D0)
    S_lo = M_ogse_mixed(td, G, N_lo, td / N_lo, tc, alpha, M0, D0)
    t_hi = np.sqrt(S_hi**2 + rn_hi**2) if rn_hi != 0.0 else S_hi
    t_lo = np.sqrt(S_lo**2 + rn_lo**2) if rn_lo != 0.0 else S_lo
    return t_hi - t_lo

# ── fitting loop ──────────────────────────────────────────────────────────────

contrast_raw_fit_store = {}
cf_raw_rows = []

for key in sorted(raw_contrast_store):
    subj, roi, direction, td = key
    G_fit = contrast_G_store[key]
    C_fit = raw_contrast_store[key]

    # ── circle markers: raw contrast values at actual data G points ───────────
    fits_td   = fit_store.get((subj, roi, direction), {}).get("fits", {}).get(td, {})
    G_hi_data = fits_td[N_hi]["G"] if N_hi in fits_td else np.array([])
    G_lo_data = fits_td[N_lo]["G"] if N_lo in fits_td else np.array([])
    C_at_Ghi  = np.interp(G_hi_data, G_fit, C_fit)
    C_at_Glo  = np.interp(G_lo_data, G_fit, C_fit)

    # ── per-N noise normalisation: RN / M0_N, matching cell 11 ────────────────
    rn    = fits_td[N_hi].get("RN", 0.0)
    rn_hi = rn / fits_td[N_hi]["M0"] if fits_td.get(N_hi, {}).get("M0", 0) > 0 else 0.0
    rn_lo = rn / fits_td[N_lo]["M0"] if fits_td.get(N_lo, {}).get("M0", 0) > 0 else 0.0

    entry = {
        "G_common": G_fit, "C_raw": C_fit,
        "G_hi": G_hi_data, "C_hi": C_at_Ghi,
        "G_lo": G_lo_data, "C_lo": C_at_Glo,
    }

    # ── free model ────────────────────────────────────────────────────────────
    _D0_f   = CF_RAW_FREE_D0_FIXED
    _C_free = _Craw_free(td, G_fit, N_hi, N_lo, 1.0, _D0_f, rn_hi, rn_lo)
    entry["free"] = {
        "D0": _D0_f, "C_curve": _C_free,
        "cost": float(0.5 * np.sum((_C_free - C_fit) ** 2)),
    }

    # ── rest model ────────────────────────────────────────────────────────────
    _D0_r = CF_RAW_REST_D0_FIXED
    if CF_RAW_REST_TC_FIXED is not None:
        _tc_r = float(CF_RAW_REST_TC_FIXED)
    else:
        def _res_rest(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_r,
                      _rn_hi=rn_hi, _rn_lo=rn_lo):
            r = _Craw_rest(_td, _G, N_hi, N_lo, np.exp(p[0]), 1.0, _D0, _rn_hi, _rn_lo) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_rest = least_squares(
            _res_rest, [np.log(CF_RAW_REST_TC_INIT)],
            bounds=([np.log(CF_RAW_REST_TC_BOUNDS[0])], [np.log(CF_RAW_REST_TC_BOUNDS[1])]),
            method="trf", max_nfev=10000,
        )
        _tc_r = float(np.exp(_r_rest.x[0]))
    _C_rest = _Craw_rest(td, G_fit, N_hi, N_lo, _tc_r, 1.0, _D0_r, rn_hi, rn_lo)
    entry["rest"] = {
        "tc": _tc_r, "D0": _D0_r, "C_curve": _C_rest,
        "cost": float(0.5 * np.sum((_C_rest - C_fit) ** 2)),
    }

    # ── tort model ────────────────────────────────────────────────────────────
    _D0_t = CF_RAW_TORT_D0_FIXED
    if CF_RAW_TORT_ALPHA_FIXED == "master":
        _al_t = _get_master_value(subj, roi, direction, CF_RAW_TORT_ALPHA_MASTER_COL)
    elif CF_RAW_TORT_ALPHA_FIXED is not None:
        _al_t = float(CF_RAW_TORT_ALPHA_FIXED)
    else:
        def _res_tort(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_t,
                      _rn_hi=rn_hi, _rn_lo=rn_lo):
            r = _Craw_tort(_td, _G, N_hi, N_lo, p[0], 1.0, _D0, _rn_hi, _rn_lo) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_tort = least_squares(
            _res_tort, [CF_RAW_TORT_ALPHA_INIT],
            bounds=([CF_RAW_TORT_ALPHA_BOUNDS[0]], [CF_RAW_TORT_ALPHA_BOUNDS[1]]),
            method="trf", max_nfev=10000,
        )
        _al_t = float(_r_tort.x[0])
    _C_tort = _Craw_tort(td, G_fit, N_hi, N_lo, _al_t, 1.0, _D0_t, rn_hi, rn_lo)
    entry["tort"] = {
        "alpha": _al_t, "D0": _D0_t, "C_curve": _C_tort,
        "cost": float(0.5 * np.sum((_C_tort - C_fit) ** 2)),
    }

    # ── mixed model ───────────────────────────────────────────────────────────
    _D0_m       = CF_RAW_MIXED_D0_FIXED
    _tc_m_fixed = float(CF_RAW_MIXED_TC_FIXED) if CF_RAW_MIXED_TC_FIXED is not None else None
    if CF_RAW_MIXED_ALPHA_FIXED == "master":
        _al_m_fixed = _get_master_value(subj, roi, direction, CF_RAW_MIXED_ALPHA_MASTER_COL)
    elif CF_RAW_MIXED_ALPHA_FIXED is not None:
        _al_m_fixed = float(CF_RAW_MIXED_ALPHA_FIXED)
    else:
        _al_m_fixed = None
    _x0_m, _lo_m, _hi_m = [], [], []
    if _tc_m_fixed is None:
        _x0_m.append(np.log(CF_RAW_MIXED_TC_INIT))
        _lo_m.append(np.log(CF_RAW_MIXED_TC_BOUNDS[0])); _hi_m.append(np.log(CF_RAW_MIXED_TC_BOUNDS[1]))
    if _al_m_fixed is None:
        _x0_m.append(CF_RAW_MIXED_ALPHA_INIT)
        _lo_m.append(CF_RAW_MIXED_ALPHA_BOUNDS[0]); _hi_m.append(CF_RAW_MIXED_ALPHA_BOUNDS[1])
    if _x0_m:
        def _res_mixed(p, _G=G_fit, _C=C_fit, _td=float(td), _D0=_D0_m,
                       _tcf=_tc_m_fixed, _alf=_al_m_fixed,
                       _rn_hi=rn_hi, _rn_lo=rn_lo):
            _i  = 0
            _tc = np.exp(p[_i]) if _tcf is None else _tcf; _i += int(_tcf is None)
            _al = p[_i]          if _alf is None else _alf
            r   = _Craw_mixed(_td, _G, N_hi, N_lo, _tc, _al, 1.0, _D0, _rn_hi, _rn_lo) - _C
            return np.where(np.isfinite(r), r, 1e6)
        _r_mixed = least_squares(
            _res_mixed, _x0_m, bounds=(_lo_m, _hi_m), method="trf", max_nfev=10000,
        )
        _i    = 0
        _tc_m = float(np.exp(_r_mixed.x[_i])) if _tc_m_fixed is None else _tc_m_fixed
        _i   += int(_tc_m_fixed is None)
        _al_m = float(_r_mixed.x[_i])           if _al_m_fixed is None else _al_m_fixed
    else:
        _tc_m = _tc_m_fixed if _tc_m_fixed is not None else CF_RAW_MIXED_TC_INIT
        _al_m = _al_m_fixed if _al_m_fixed is not None else CF_RAW_MIXED_ALPHA_INIT
    _C_mixed = _Craw_mixed(td, G_fit, N_hi, N_lo, _tc_m, _al_m, 1.0, _D0_m, rn_hi, rn_lo)
    entry["mixed"] = {
        "tc": _tc_m, "alpha": _al_m, "D0": _D0_m, "C_curve": _C_mixed,
        "cost": float(0.5 * np.sum((_C_mixed - C_fit) ** 2)),
    }

    contrast_raw_fit_store[key] = entry

    print(
        f"{subj:6s}  {roi:25s}  {direction}  td={td:5.0f} ms  "
        f"rest tc={entry['rest']['tc']:.3f} ms  "
        f"tort α={entry['tort']['alpha']:.3f}  "
        f"mixed tc={entry['mixed']['tc']:.3f} ms α={entry['mixed']['alpha']:.3f}"
    )
    cf_raw_rows.append(dict(
        subj=subj, roi=roi, direction=direction, td_ms=td,
        free_cost   =entry["free"]["cost"],
        rest_tc_ms  =entry["rest"]["tc"],    rest_cost  =entry["rest"]["cost"],
        tort_alpha  =entry["tort"]["alpha"], tort_cost  =entry["tort"]["cost"],
        mixed_tc_ms =entry["mixed"]["tc"],   mixed_alpha=entry["mixed"]["alpha"],
        mixed_cost  =entry["mixed"]["cost"],
    ))

cf_raw_df = pd.DataFrame(cf_raw_rows)
_cf_raw_xlsx = OUT_DIR / CF_RAW_OUT_SUBDIR / "contrast_raw_fit_results.xlsx"
cf_raw_df.to_excel(_cf_raw_xlsx, index=False)
print(f"\nSaved {len(cf_raw_df)} rows → {_cf_raw_xlsx}")
cf_raw_df

BRAIN   AntCC                      long  td=   76 ms  rest tc=361.634 ms  tort α=0.997  mixed tc=138.296 ms α=0.883
BRAIN   AntCC                      long  td=   90 ms  rest tc=315.664 ms  tort α=0.999  mixed tc=110.867 ms α=0.906
BRAIN   AntCC                      long  td=  120 ms  rest tc=2.000 ms  tort α=0.500  mixed tc=2.000 ms α=0.500
BRAIN   AntCC                      long  td=  143 ms  rest tc=13.232 ms  tort α=0.563  mixed tc=13.232 ms α=0.000
BRAIN   AntCC                      long  td=  210 ms  rest tc=511.248 ms  tort α=0.997  mixed tc=182.073 ms α=0.909
BRAIN   AntCC                      tra  td=   76 ms  rest tc=341.489 ms  tort α=0.997  mixed tc=133.019 ms α=0.887
BRAIN   AntCC                      tra  td=   90 ms  rest tc=318.231 ms  tort α=0.999  mixed tc=109.893 ms α=0.906
BRAIN   AntCC                      tra  td=  120 ms  rest tc=2.000 ms  tort α=0.500  mixed tc=2.000 ms α=0.500
BRAIN   AntCC                      tra  td=  143 ms  rest tc=2.000 ms  tort α=0.500  

/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:101: RuntimeWarning: overflow encountered in exp
  - np.exp(x / tc) / 2
/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:117: RuntimeWarning: invalid value encountered in multiply
  phi_cross = 2 * tc**2 * inner_cross * bSE**2 / (np.exp(x / tc) + 1)
/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:117: RuntimeWarning: overflow encountered in exp
  phi_cross = 2 * tc**2 * inner_cross * bSE**2 / (np.exp(x / tc) + 1)
/mnt/storage/tier2/MUMI-EXT-001/mumi-data/Project-Balseiro-Microstructure/nogse_pipeline/notebooks-portable/portable_helpers.py:117: RuntimeWarning: invalid value encountered in divide
  phi_cross = 2 * tc**2 * inner_cross * bSE**2 / (np.exp(x / tc) + 1)


BRAIN   Left-Lateral-Ventricle     tra  td=  143 ms  rest tc=230.757 ms  tort α=0.960  mixed tc=0.051 ms α=0.960
BRAIN   Left-Lateral-Ventricle     tra  td=  210 ms  rest tc=106.397 ms  tort α=1.000  mixed tc=106.396 ms α=0.000
BRAIN   MidAntCC                   long  td=   76 ms  rest tc=374.238 ms  tort α=0.997  mixed tc=111.990 ms α=0.857
BRAIN   MidAntCC                   long  td=   90 ms  rest tc=309.014 ms  tort α=0.999  mixed tc=114.509 ms α=0.903
BRAIN   MidAntCC                   long  td=  120 ms  rest tc=334.584 ms  tort α=0.999  mixed tc=121.415 ms α=0.901
BRAIN   MidAntCC                   long  td=  143 ms  rest tc=335.051 ms  tort α=0.997  mixed tc=131.986 ms α=0.902
BRAIN   MidAntCC                   long  td=  210 ms  rest tc=465.548 ms  tort α=0.997  mixed tc=178.618 ms α=0.908
BRAIN   MidAntCC                   tra  td=   76 ms  rest tc=324.862 ms  tort α=0.997  mixed tc=109.290 ms α=0.857
BRAIN   MidAntCC                   tra  td=   90 ms  rest tc=312.013 ms  tort

,subj,roi,direction,td_ms,free_cost,rest_tc_ms,rest_cost,tort_alpha,tort_cost,mixed_tc_ms,mixed_alpha,mixed_cost
0,BRAIN,AntCC,long,76.0,2.801688e-14,361.633607,6.085329e-10,0.996882,3.228841e-09,138.295910,8.832521e-01,3.158339e-10
1,BRAIN,AntCC,long,90.0,8.721710e-14,315.664265,1.731636e-09,0.998750,1.959726e-09,110.866789,9.057251e-01,7.541394e-10
2,BRAIN,AntCC,long,120.0,inf,2.000000,inf,0.500000,inf,2.000000,5.000000e-01,inf
3,BRAIN,AntCC,long,143.4,2.462772e-04,13.231681,5.325399e-06,0.563462,1.092582e-04,13.231691,5.866975e-10,5.325399e-06
4,BRAIN,AntCC,long,210.0,4.197311e-18,511.248241,2.007102e-09,0.996750,4.846736e-09,182.072546,9.087558e-01,7.016834e-10
...,...,...,...,...,...,...,...,...,...,...,...,...
175,MBBL,Right-Lateral-Ventricle,long,210.0,1.470309e-02,109.597003,1.047569e-02,1.000000,1.470309e-02,109.590632,8.195016e-09,1.047569e-02
176,MBBL,Right-Lateral-Ventricle,tra,90.0,2.340465e-02,38.272331,8.299532e-04,0.943816,1.609917e-02,38.272294,2.957266e-10,8.299532e-04
177,MBBL,Right-Lateral-Ventricle,tra,120.0,1.496728e-02,54.084878,6.350068e-04,0.965823,8.927213e-03,9.611773,8.899878e-01,4.445536e-05
178,MBBL,Right-Lateral-Ventricle,tra,143.4,1.684946e-03,118.205089,2.639109e-04,0.988718,8.132806e-04,9.862915,9.647341e-01,9.387743e-05


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT CONTRAST FITS (RAW)  —  per (subj, roi, td)
# One figure per (subj, roi, td); columns = directions
# Data: raw_contrast values at N_hi G points (filled ●) and N_lo G points (empty ○)
# Fits: free (C0), rest (C1), tort (C2), mixed (C3)
# ══════════════════════════════════════════════════════════════════════════════

_cf_raw_subjs = sorted(set(k[0] for k in contrast_raw_fit_store))
_cf_raw_rois  = sorted(set(k[1] for k in contrast_raw_fit_store))
_cf_raw_tds   = sorted(set(k[3] for k in contrast_raw_fit_store))

for subj in _cf_raw_subjs:
    for roi in _cf_raw_rois:
        for td in _cf_raw_tds:
            dirs_here = [d for d in DIRECTIONS if (subj, roi, d, td) in contrast_raw_fit_store]
            if not dirs_here:
                continue
            fig, axes = plt.subplots(
                1, len(dirs_here),
                figsize=(4.5 * len(dirs_here), 4),
                sharey=True, squeeze=False,
            )
            axes = axes[0]

            for ax, direction in zip(axes, dirs_here):
                entry = contrast_raw_fit_store[(subj, roi, direction, td)]
                G_c   = entry["G_common"]

                # ── data markers (raw contrast at actual G points) ─────────────
                ax.scatter(entry["G_hi"], entry["C_hi"],
                           color="k", s=20, zorder=5, label=f"N={N_hi} pts")
                ax.scatter(entry["G_lo"], entry["C_lo"],
                           facecolors="none", edgecolors="k", s=60, zorder=5,
                           label=f"N={N_lo} pts")

                # ── model fit lines ───────────────────────────────────────────
                ax.plot(G_c, entry["free"]["C_curve"],  color="C0", lw=1.5, label="free")
                ax.plot(G_c, entry["rest"]["C_curve"],  color="C1", lw=1.5,
                        label=f"rest  tc={entry['rest']['tc']:.2f} ms")
                ax.plot(G_c, entry["tort"]["C_curve"],  color="C2", lw=1.5,
                        label=f"tort  α={entry['tort']['alpha']:.3f}")
                ax.plot(G_c, entry["mixed"]["C_curve"], color="C3", lw=1.5,
                        label=f"mixed  tc={entry['mixed']['tc']:.2f} ms, α={entry['mixed']['alpha']:.3f}")

                ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
                ax.set_title(direction, fontsize=9)
                ax.set_xlabel(G_LABEL)
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
                ax.set_axisbelow(True)
                ax.legend(fontsize=6, loc="best")

            axes[0].set_ylabel(f"ΔS_raw (N={N_hi}−N={N_lo}) [a.u.]")
            fig.suptitle(
                f"{subj}  |  {roi}  |  td={td:.0f} ms  —  raw contrast fits", fontsize=10
            )
            fig.tight_layout()
            fig.savefig(
                OUT_DIR / CF_RAW_OUT_SUBDIR / f"contrast_raw_fit_{subj}_{roi}_{td:.0f}.png",
                dpi=120,
            )
            plt.close(fig)

print("Raw contrast fit plots saved.")

Raw contrast fit plots saved.
